# Notebook 19 — Observable-matched particle POD and identity audit

## Aim

This notebook tests the strongest remaining representation question from the route-order comparison.

The existing labelled particle states contain all particle coordinates,

$$
X = \left(X_s,X_\ell\right),
$$

where $X_s$ and $X_\ell$ denote the small- and large-species coordinates. However, the evaluated small-species volume-fraction field depends only on the small-particle $x,z$ positions:

$$
C_h(X)=C_h\!\left(X_{s,XZ}\right).
$$

Notebook 13 removed the unused axial coordinate $y$ but retained both particle species. The present experiment removes both sources of information that do not enter the evaluated observable and applies POD only to the labelled small-species $XZ$ coordinates.

## Frozen comparison

The experiment preserves:

- the five Notebook 12 particle-size-ratio cases;
- 101 first-cycle training snapshots;
- 99 second-cycle validation snapshots;
- ranks $r\in\{0,8,14,27,100\}$;
- the $100\times100$ cell-centred continuum grid;
- the implemented two-dimensional Lucy kernel with support $h=3$;
- axial averaging length $W=10$;
- the existing small-species volume convention;
- truth-projected validation coefficients;
- the existing space-time and physical field diagnostics.

The comparison is between:

1. continuum-field POD followed by field reconstruction;
2. all-particle labelled $XYZ$ POD followed by coarse-graining;
3. all-particle labelled $XZ$ POD followed by coarse-graining;
4. small-species-only labelled $XZ$ POD followed by coarse-graining.

No result is prescribed in advance. A large improvement would revise the interpretation of the previous particle-coordinate result; persistently large error would close the strongest observable-relevance objection.

## Particle-row continuity diagnostic

The compact caches do not contain persistent particle identifiers. This notebook therefore measures consecutive-snapshot displacement distributions under the stored row correspondence and compares them with nearest same-species displacement scales.

This is a diagnostic for conspicuous ordering discontinuities. Smooth row trajectories can support the stored-order assumption, but they cannot prove physical identity among otherwise identical particles.

## Scope

This notebook:

- uses only the existing compact particle caches and accepted result manifests;
- does not access raw DEM particle files;
- does not generate new DEM simulations;
- does not train a predictive surrogate;
- does not use the Chapter 5 external evaluation cases;
- does not alter Notebooks 11–18 or their accepted result directories.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def n19_locate_repository_root(start):
    start = Path(start).resolve()

    for candidate in [
        start,
        *start.parents,
    ]:
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "results").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate the repository root from "
        f"{start}."
    )


def n19_sha256(path, chunk_size=8 * 1024**2):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


N19_REPO_ROOT = n19_locate_repository_root(
    Path.cwd()
)

N19_RANKS = (
    0,
    8,
    14,
    27,
    100,
)

N19_SMALL_SPECIES = 1
N19_STATIONARY_TIME_MIN = 1377.0
N19_COMPARISON_TIME_MAX = 1720.72
N19_LUCY_CUTOFF = 3.0
N19_AXIAL_AVERAGING_LENGTH = 10.0
N19_NEGATIVE_TOLERANCE = -1.0e-4


N19_NOTEBOOK12_TABLE_DIRECTORY = (
    N19_REPO_ROOT
    / "results"
    / "tables"
    / "multi_parameter_route_order"
)

N19_NOTEBOOK12_MANIFEST_PATH = (
    N19_NOTEBOOK12_TABLE_DIRECTORY
    / "multi_parameter_route_order_manifest.json"
)

N19_NOTEBOOK13_TABLE_DIRECTORY = (
    N19_REPO_ROOT
    / "results"
    / "tables"
    / "xz_particle_representation_sensitivity"
)

N19_NOTEBOOK13_MANIFEST_PATH = (
    N19_NOTEBOOK13_TABLE_DIRECTORY
    / "xz_particle_representation_sensitivity_manifest.json"
)

N19_REFERENCE_FIELD_PATH = (
    N19_REPO_ROOT
    / "data"
    / "raw"
    / "CG_field_phi_small"
    / "rotating_drum_bidisperse_size_ratio_1_000000_num_1.stat"
)

N19_PILOT_CACHE_PATH = (
    N19_REPO_ROOT
    / "data"
    / "interim"
    / "particle_positions_ratio_1_000000_num_1.npz"
)

N19_PILOT_CACHE_MANIFEST_PATH = (
    N19_REPO_ROOT
    / "results"
    / "tables"
    / "particle_snapshot_representation_ratio_1_000000_num_1.json"
)

N19_MULTI_PARAMETER_CACHE_DIRECTORY = (
    N19_REPO_ROOT
    / "data"
    / "interim"
    / "multi_parameter_route_order"
)


# These directories are defined now but are not created
# until the scientific calculations pass.
N19_TABLE_DIRECTORY = (
    N19_REPO_ROOT
    / "results"
    / "tables"
    / "observable_matched_particle_pod"
)

N19_FIGURE_DIRECTORY = (
    N19_REPO_ROOT
    / "results"
    / "figures"
    / "observable_matched_particle_pod"
)


n19_required_inputs = pd.DataFrame(
    [
        {
            "input": "Notebook 12 manifest",
            "path": N19_NOTEBOOK12_MANIFEST_PATH,
        },
        {
            "input": "Notebook 13 manifest",
            "path": N19_NOTEBOOK13_MANIFEST_PATH,
        },
        {
            "input": "pilot compact cache",
            "path": N19_PILOT_CACHE_PATH,
        },
        {
            "input": "pilot compact-cache manifest",
            "path": N19_PILOT_CACHE_MANIFEST_PATH,
        },
        {
            "input": "reference MercuryCG field",
            "path": N19_REFERENCE_FIELD_PATH,
        },
        {
            "input": "multi-parameter cache directory",
            "path": N19_MULTI_PARAMETER_CACHE_DIRECTORY,
        },
    ]
)

n19_required_inputs["exists"] = [
    path.exists()
    for path in n19_required_inputs["path"]
]


display(
    n19_required_inputs.assign(
        path=n19_required_inputs[
            "path"
        ].astype(str)
    )
)

assert n19_required_inputs["exists"].all()
assert N19_RANKS == (0, 8, 14, 27, 100)


with N19_NOTEBOOK12_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    N19_NOTEBOOK12_MANIFEST = json.load(handle)


with N19_NOTEBOOK13_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    N19_NOTEBOOK13_MANIFEST = json.load(handle)


assert (
    N19_NOTEBOOK12_MANIFEST["schema_version"]
    == 1
)

assert (
    N19_NOTEBOOK13_MANIFEST["schema_version"]
    == 1
)

assert (
    N19_NOTEBOOK12_MANIFEST[
        "experiment"
    ]["case_count"]
    == 5
)

assert (
    N19_NOTEBOOK13_MANIFEST[
        "experiment"
    ]["case_count"]
    == 5
)

assert tuple(
    N19_NOTEBOOK12_MANIFEST[
        "experiment"
    ]["ranks"]
) == N19_RANKS

assert tuple(
    N19_NOTEBOOK13_MANIFEST[
        "experiment"
    ]["ranks"]
) == N19_RANKS


n19_notebook13_artifact_audit = []

for record_group in [
    "case_manifests",
    "aggregate_artifacts",
]:
    for artifact_name, record in (
        N19_NOTEBOOK13_MANIFEST[
            record_group
        ].items()
    ):
        artifact_path = (
            N19_REPO_ROOT
            / record["relative_path"]
        )

        exists = artifact_path.is_file()

        hash_matches = (
            exists
            and n19_sha256(artifact_path)
            == record["sha256"]
        )

        n19_notebook13_artifact_audit.append(
            {
                "record_group": record_group,
                "artifact": artifact_name,
                "path": str(artifact_path),
                "exists": exists,
                "hash_matches": hash_matches,
            }
        )


N19_NOTEBOOK13_ARTIFACT_AUDIT = (
    pd.DataFrame(
        n19_notebook13_artifact_audit
    )
)


display(
    N19_NOTEBOOK13_ARTIFACT_AUDIT
)


assert (
    N19_NOTEBOOK13_ARTIFACT_AUDIT[
        "exists"
    ].all()
)

assert (
    N19_NOTEBOOK13_ARTIFACT_AUDIT[
        "hash_matches"
    ].all()
)


print(
    "Python executable:",
    sys.executable,
)

print(
    "Repository root:",
    N19_REPO_ROOT,
)

print(
    "Notebook 12 manifest SHA-256:",
    n19_sha256(
        N19_NOTEBOOK12_MANIFEST_PATH
    ),
)

print(
    "Notebook 13 manifest SHA-256:",
    n19_sha256(
        N19_NOTEBOOK13_MANIFEST_PATH
    ),
)

print(
    "Frozen ranks:",
    N19_RANKS,
)

print(
    "Small species:",
    N19_SMALL_SPECIES,
)

print(
    "Grid, Lucy support and width:",
    (100, 100),
    N19_LUCY_CUTOFF,
    N19_AXIAL_AVERAGING_LENGTH,
)

print(
    "Numerical particle arrays loaded:",
    False,
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "Files written:",
    False,
)

print(
    "Notebook 19 read-only preflight passed:",
    True,
)

,input,path,exists
0,Notebook 12 manifest,/Users/rallen/Documents/msc-rom-particle-syste...,True
1,Notebook 13 manifest,/Users/rallen/Documents/msc-rom-particle-syste...,True
2,pilot compact cache,/Users/rallen/Documents/msc-rom-particle-syste...,True
3,pilot compact-cache manifest,/Users/rallen/Documents/msc-rom-particle-syste...,True
4,reference MercuryCG field,/Users/rallen/Documents/msc-rom-particle-syste...,True
5,multi-parameter cache directory,/Users/rallen/Documents/msc-rom-particle-syste...,True


,record_group,artifact,path,exists,hash_matches
0,case_manifests,ratio_1_000000_num_1,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
1,case_manifests,ratio_1_252525_num_26,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
2,case_manifests,ratio_1_494949_num_50,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
3,case_manifests,ratio_1_747475_num_75,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
4,case_manifests,ratio_2_000000_num_100,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
5,aggregate_artifacts,master_field_csv,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
6,aggregate_artifacts,master_particle_csv,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
7,aggregate_artifacts,master_energy_csv,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
8,aggregate_artifacts,comparison_by_rank_csv,/Users/rallen/Documents/msc-rom-particle-syste...,True,True
9,aggregate_artifacts,rank_100_summary_csv,/Users/rallen/Documents/msc-rom-particle-syste...,True,True


Python executable: /Users/rallen/Documents/msc-rom-particle-systems/.venv/bin/python
Repository root: /Users/rallen/Documents/msc-rom-particle-systems
Notebook 12 manifest SHA-256: f093f7de958f5fcd6860ebcfee8416af27aa01d4b1c8ed1488e05d7fb8e9e844
Notebook 13 manifest SHA-256: 389bb1a53975fe098c7da908b5886adfb42f13237f6c6464b3c170f7b6d0cfce
Frozen ranks: (0, 8, 14, 27, 100)
Small species: 1
Grid, Lucy support and width: (100, 100) 3.0 10.0
Numerical particle arrays loaded: False
Raw DEM files accessed: False
Files written: False
Notebook 19 read-only preflight passed: True


## Frozen inputs and five-case plan

The Notebook 12 and Notebook 13 manifest hashes are now frozen as direct inputs to this experiment.

This stage verifies every accepted Notebook 12 case manifest and every compact particle-cache manifest. It records:

- particle and small-species counts;
- cache shapes, sizes and SHA-256 hashes;
- physical cycle boundaries;
- training and validation counts;
- the numerical ranks previously obtained for the continuum and all-particle coordinate representations.

No numerical particle array is loaded at this stage.

In [2]:
N19_EXPECTED_NOTEBOOK12_MANIFEST_SHA256 = (
    "f093f7de958f5fcd6860ebcfee8416af27aa01d4b1c8ed1488e05d7fb8e9e844"
)

N19_EXPECTED_NOTEBOOK13_MANIFEST_SHA256 = (
    "389bb1a53975fe098c7da908b5886adfb42f13237f6c6464b3c170f7b6d0cfce"
)


assert (
    n19_sha256(
        N19_NOTEBOOK12_MANIFEST_PATH
    )
    == N19_EXPECTED_NOTEBOOK12_MANIFEST_SHA256
)

assert (
    n19_sha256(
        N19_NOTEBOOK13_MANIFEST_PATH
    )
    == N19_EXPECTED_NOTEBOOK13_MANIFEST_SHA256
)


n19_case_records = []


for case_id, manifest_record in (
    N19_NOTEBOOK12_MANIFEST[
        "case_manifests"
    ].items()
):
    result_manifest_path = (
        N19_REPO_ROOT
        / manifest_record["relative_path"]
    )

    assert result_manifest_path.is_file(), (
        f"Missing result manifest: "
        f"{result_manifest_path}"
    )

    result_manifest_hash = n19_sha256(
        result_manifest_path
    )

    assert (
        result_manifest_hash
        == manifest_record["sha256"]
    ), (
        f"Notebook 12 case-manifest hash "
        f"mismatch: {case_id}"
    )


    with result_manifest_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        result_manifest = json.load(handle)


    assert (
        result_manifest["schema_version"]
        == 1
    )

    configuration = (
        result_manifest["configuration"]
    )

    case_metadata = (
        result_manifest["case"]
    )

    validation_metadata = (
        result_manifest["validation"]
    )


    assert tuple(
        configuration["ranks"]
    ) == N19_RANKS

    assert np.isclose(
        configuration[
            "stationary_time_min"
        ],
        N19_STATIONARY_TIME_MIN,
        rtol=0.0,
        atol=1.0e-12,
    )

    assert np.isclose(
        configuration[
            "comparison_time_max"
        ],
        N19_COMPARISON_TIME_MAX,
        rtol=0.0,
        atol=1.0e-12,
    )

    assert np.isclose(
        configuration[
            "lucy_cutoff"
        ],
        N19_LUCY_CUTOFF,
        rtol=0.0,
        atol=1.0e-12,
    )

    assert (
        configuration["small_species"]
        == N19_SMALL_SPECIES
    )


    n19_case_records.append(
        {
            "case_id": case_id,
            "size_ratio": float(
                case_metadata[
                    "size_ratio"
                ]
            ),
            "simulation_number": int(
                case_metadata[
                    "simulation_number"
                ]
            ),
            "particle_count": int(
                case_metadata[
                    "particle_count"
                ]
            ),
            "small_particle_count": int(
                case_metadata[
                    "small_particle_count"
                ]
            ),
            "large_particle_count": (
                int(
                    case_metadata[
                        "particle_count"
                    ]
                )
                - int(
                    case_metadata[
                        "small_particle_count"
                    ]
                )
            ),
            "training_snapshot_count": int(
                case_metadata[
                    "training_snapshot_count"
                ]
            ),
            "validation_snapshot_count": int(
                case_metadata[
                    "validation_snapshot_count"
                ]
            ),
            "continuum_numerical_rank": int(
                validation_metadata[
                    "continuum_numerical_rank"
                ]
            ),
            "xyz_particle_numerical_rank": int(
                validation_metadata[
                    "particle_numerical_rank"
                ]
            ),
            "result_manifest_path":
                result_manifest_path,
            "result_manifest_sha256":
                result_manifest_hash,
        }
    )


N19_CASE_PLAN = (
    pd.DataFrame(
        n19_case_records
    )
    .sort_values(
        "size_ratio"
    )
    .reset_index(
        drop=True
    )
)


assert len(N19_CASE_PLAN) == 5
assert N19_CASE_PLAN["case_id"].is_unique

assert (
    N19_CASE_PLAN[
        "training_snapshot_count"
    ] == 101
).all()

assert (
    N19_CASE_PLAN[
        "validation_snapshot_count"
    ] == 99
).all()

assert (
    N19_CASE_PLAN[
        "continuum_numerical_rank"
    ] == 100
).all()

assert (
    N19_CASE_PLAN[
        "xyz_particle_numerical_rank"
    ] == 100
).all()


n19_compact_records = []


for case_row in (
    N19_CASE_PLAN.itertuples(
        index=False
    )
):
    case_id = str(
        case_row.case_id
    )

    if (
        case_id
        == "ratio_1_000000_num_1"
    ):
        compact_manifest_path = (
            N19_PILOT_CACHE_MANIFEST_PATH
        )

        expected_schema_version = 1

    else:
        compact_manifest_path = (
            N19_MULTI_PARAMETER_CACHE_DIRECTORY
            / (
                "snapshot_manifest_"
                f"{case_id}.json"
            )
        )

        expected_schema_version = 2


    assert compact_manifest_path.is_file(), (
        f"Missing compact manifest: "
        f"{compact_manifest_path}"
    )


    with compact_manifest_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        compact_manifest = json.load(handle)


    assert (
        compact_manifest[
            "schema_version"
        ]
        == expected_schema_version
    )


    compact_artifact_record = (
        compact_manifest[
            "compact_artifact"
        ]
    )

    compact_path = (
        N19_REPO_ROOT
        / compact_artifact_record[
            "relative_path"
        ]
    )


    assert compact_path.is_file(), (
        f"Missing compact cache: "
        f"{compact_path}"
    )


    observed_compact_size = (
        compact_path.stat().st_size
    )

    observed_compact_hash = (
        n19_sha256(
            compact_path
        )
    )


    assert (
        observed_compact_size
        == compact_artifact_record[
            "size_bytes"
        ]
    ), (
        f"Compact-cache size mismatch: "
        f"{case_id}"
    )

    assert (
        observed_compact_hash
        == compact_artifact_record[
            "sha256"
        ]
    ), (
        f"Compact-cache hash mismatch: "
        f"{case_id}"
    )


    if expected_schema_version == 1:
        tensor_shape = tuple(
            compact_manifest[
                "particle_representation"
            ]["tensor_shape"]
        )

        stationary_threshold = float(
            compact_manifest[
                "snapshot_selection"
            ]["stationary_threshold"]
        )

        cycle_boundary_time = float(
            compact_manifest[
                "physical_cycle_split"
            ]["cycle_boundary_time"]
        )

    else:
        tensor_shape = tuple(
            compact_manifest[
                "particle_representation"
            ]["shape"]
        )

        stationary_threshold = float(
            compact_manifest[
                "configuration"
            ]["stationary_time_min"]
        )

        cycle_boundary_time = float(
            compact_manifest[
                "cycle_split"
            ]["cycle_boundary_time"]
        )

        assert np.isclose(
            compact_manifest[
                "configuration"
            ]["comparison_time_max"],
            N19_COMPARISON_TIME_MAX,
            rtol=0.0,
            atol=1.0e-12,
        )


    assert np.isclose(
        stationary_threshold,
        N19_STATIONARY_TIME_MIN,
        rtol=0.0,
        atol=1.0e-12,
    )

    assert tensor_shape == (
        201,
        int(
            case_row.particle_count
        ),
        3,
    )


    if (
        case_id
        == "ratio_1_000000_num_1"
    ):
        assert (
            compact_path.resolve()
            == N19_PILOT_CACHE_PATH.resolve()
        )


    n19_compact_records.append(
        {
            "case_id": case_id,
            "compact_path": compact_path,
            "compact_manifest_path":
                compact_manifest_path,
            "compact_manifest_schema":
                expected_schema_version,
            "tensor_shape": tensor_shape,
            "cycle_boundary_time":
                cycle_boundary_time,
            "compressed_size_MiB": (
                observed_compact_size
                / 1024**2
            ),
            "compact_sha256":
                observed_compact_hash,
            "size_matches": True,
            "hash_matches": True,
        }
    )


N19_COMPACT_AUDIT = (
    pd.DataFrame(
        n19_compact_records
    )
)


N19_CASE_PLAN = (
    N19_CASE_PLAN.merge(
        N19_COMPACT_AUDIT,
        on="case_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "size_ratio"
    )
    .reset_index(
        drop=True
    )
)


assert (
    N19_CASE_PLAN[
        "compact_path"
    ].notna().all()
)

assert (
    N19_CASE_PLAN[
        "size_matches"
    ].all()
)

assert (
    N19_CASE_PLAN[
        "hash_matches"
    ].all()
)


display_columns = [
    "case_id",
    "size_ratio",
    "particle_count",
    "small_particle_count",
    "large_particle_count",
    "tensor_shape",
    "cycle_boundary_time",
    "compressed_size_MiB",
    "hash_matches",
]


print(
    "Frozen five-case plan:"
)

display(
    N19_CASE_PLAN[
        display_columns
    ].style.format(
        {
            "size_ratio": "{:.6f}",
            "cycle_boundary_time":
                "{:.9f}",
            "compressed_size_MiB":
                "{:.2f}",
        }
    )
)


print(
    "Total compact-cache size:",
    (
        f"{N19_CASE_PLAN['compressed_size_MiB'].sum():.2f} MiB"
    ),
)

print(
    "Five compact-cache hashes verified:",
    bool(
        N19_CASE_PLAN[
            "hash_matches"
        ].all()
    ),
)

print(
    "Training/validation snapshots per case:",
    101,
    99,
)

print(
    "Numerical particle arrays loaded:",
    False,
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "Files written:",
    False,
)

print(
    "Notebook 19 frozen-input audit passed:",
    True,
)

Frozen five-case plan:


,case_id,size_ratio,particle_count,small_particle_count,large_particle_count,tensor_shape,cycle_boundary_time,compressed_size_MiB,hash_matches
0,ratio_1_000000_num_1,1.000000,10792,5396,5396,"(201, 10792, 3)",1550.205479286,47.96,True
1,ratio_1_252525_num_26,1.252525,11626,7701,3925,"(201, 11626, 3)",1550.301982286,51.65,True
2,ratio_1_494949_num_50,1.494949,13599,10462,3137,"(201, 13599, 3)",1550.354969286,60.42,True
3,ratio_1_747475_num_75,1.747475,16584,13961,2623,"(201, 16584, 3)",1550.263075286,73.68,True
4,ratio_2_000000_num_100,2.000000,20449,18169,2280,"(201, 20449, 3)",1550.299681286,90.84,True


Total compact-cache size: 324.54 MiB
Five compact-cache hashes verified: True
Training/validation snapshots per case: 101 99
Numerical particle arrays loaded: False
Raw DEM files accessed: False
Files written: False
Notebook 19 frozen-input audit passed: True


## Pilot-case loading and physical conventions

Before constructing a new POD, the pilot compact cache is loaded and checked against the frozen metadata.

For each snapshot, let

$$
X_t \in \mathbb{R}^{N\times 3}
$$

denote the labelled particle-coordinate array. The small-particle subset is

$$
X_{s,t}
=
\left\{
X_{t,j}:s_j=1
\right\},
$$

and its observable-matched representation retains only the radial-plane coordinates,

$$
X_{s,XZ,t}
=
X_{s,t}[:,(x,z)]
\in
\mathbb{R}^{N_s\times 2}.
$$

The particles have the mild radius polydispersity used by the supplied DEM simulations. A characteristic small-particle diameter is therefore defined from the median stored small-particle radius,

$$
d_s
=
2\,\operatorname{median}_{j:s_j=1}(r_j).
$$

The complete minimum-to-maximum radius range is retained in the audit; the median is used only to report a representative dimensionless kernel-support ratio.

The ratio

$$
\frac{h}{d_s}
$$

then expresses the Lucy-kernel support radius relative to the small-particle diameter. This is recorded explicitly because the apparent smoothing strength depends on both the dimensional cutoff $h$ and the particle scale.

This cell loads only the equal-size pilot case. It verifies the tensor dimensions, species counts, radii, time ordering and frozen training/validation split before any POD or coarse-graining calculation is performed.

In [3]:
def n19_load_compact_case(
    case_row,
):
    """Load and validate one frozen compact particle case."""
    case_id = str(
        case_row["case_id"]
    )

    compact_path = Path(
        case_row["compact_path"]
    )

    compact_manifest_path = Path(
        case_row["compact_manifest_path"]
    )

    assert compact_path.is_file()
    assert compact_manifest_path.is_file()

    assert (
        n19_sha256(compact_path)
        == case_row["compact_sha256"]
    )

    with compact_manifest_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        compact_manifest = json.load(handle)

    with np.load(
        compact_path,
        allow_pickle=False,
    ) as compact_data:
        required_arrays = {
            "positions_xyz",
            "snapshot_times",
            "particle_radii",
            "species_index",
        }

        missing_arrays = (
            required_arrays
            - set(compact_data.files)
        )

        assert not missing_arrays, (
            f"{case_id} is missing arrays: "
            f"{sorted(missing_arrays)}"
        )

        compact_keys = sorted(
            compact_data.files
        )

        positions_xyz = np.asarray(
            compact_data["positions_xyz"],
            dtype=np.float64,
        ).copy()

        snapshot_times = np.asarray(
            compact_data["snapshot_times"],
            dtype=np.float64,
        ).copy()

        particle_radii = np.asarray(
            compact_data["particle_radii"],
            dtype=np.float64,
        ).copy()

        species_index = np.asarray(
            compact_data["species_index"],
            dtype=np.int64,
        ).copy()

        if (
            "source_snapshot_numbers"
            in compact_data.files
        ):
            source_snapshot_numbers = np.asarray(
                compact_data[
                    "source_snapshot_numbers"
                ],
                dtype=np.int64,
            ).copy()
        else:
            source_snapshot_numbers = np.arange(
                positions_xyz.shape[0],
                dtype=np.int64,
            )

        if "cycle_labels" in compact_data.files:
            cycle_labels = np.asarray(
                compact_data["cycle_labels"]
            ).astype(str).copy()
        else:
            cycle_labels = None

    expected_shape = tuple(
        case_row["tensor_shape"]
    )

    assert positions_xyz.shape == expected_shape
    assert snapshot_times.shape == (
        expected_shape[0],
    )
    assert particle_radii.shape == (
        expected_shape[1],
    )
    assert species_index.shape == (
        expected_shape[1],
    )
    assert source_snapshot_numbers.shape == (
        expected_shape[0],
    )

    assert np.isfinite(
        positions_xyz
    ).all()

    assert np.isfinite(
        snapshot_times
    ).all()

    assert np.isfinite(
        particle_radii
    ).all()

    assert np.all(
        particle_radii > 0.0
    )

    assert np.all(
        np.diff(snapshot_times) > 0.0
    )

    small_mask = (
        species_index
        == N19_SMALL_SPECIES
    )

    large_mask = ~small_mask

    observed_small_count = int(
        small_mask.sum()
    )

    observed_large_count = int(
        large_mask.sum()
    )

    assert observed_small_count == int(
        case_row["small_particle_count"]
    )

    assert observed_large_count == int(
        case_row["large_particle_count"]
    )

    assert (
        observed_small_count
        + observed_large_count
        == positions_xyz.shape[1]
    )

    small_radii = particle_radii[
        small_mask
    ]

    large_radii = particle_radii[
        large_mask
    ]

    small_radius = float(
        np.median(small_radii)
    )

    large_radius = float(
        np.median(large_radii)
    )

    assert np.allclose(
        small_radii,
        small_radius,
        rtol=6.0e-2,
        atol=0.0,
    )

    assert np.allclose(
        large_radii,
        large_radius,
        rtol=6.0e-2,
        atol=0.0,
    )

    small_diameter = (
        2.0 * small_radius
    )

    large_diameter = (
        2.0 * large_radius
    )

    observed_size_ratio = (
        large_diameter
        / small_diameter
    )

    assert np.isclose(
        observed_size_ratio,
        float(case_row["size_ratio"]),
        rtol=2.0e-2,
        atol=0.0,
    )

    lucy_support_to_small_diameter = (
        N19_LUCY_CUTOFF
        / small_diameter
    )

    comparison_mask = (
        snapshot_times
        <= (
            N19_COMPARISON_TIME_MAX
            + 1.0e-8
        )
    )

    cycle_boundary_time = float(
        case_row["cycle_boundary_time"]
    )

    training_indices = np.flatnonzero(
        comparison_mask
        & (
            snapshot_times
            <= cycle_boundary_time
        )
    )

    validation_indices = np.flatnonzero(
        comparison_mask
        & (
            snapshot_times
            > cycle_boundary_time
        )
    )

    excluded_indices = np.flatnonzero(
        ~comparison_mask
    )

    assert len(training_indices) == int(
        case_row["training_snapshot_count"]
    )

    assert len(validation_indices) == int(
        case_row["validation_snapshot_count"]
    )

    assert np.intersect1d(
        training_indices,
        validation_indices,
    ).size == 0

    assert (
        len(training_indices)
        + len(validation_indices)
        + len(excluded_indices)
        == len(snapshot_times)
    )

    positions_small_xyz = positions_xyz[
        :,
        small_mask,
        :,
    ]

    positions_small_xz = positions_small_xyz[
        :,
        :,
        [0, 2],
    ]

    assert positions_small_xyz.shape == (
        expected_shape[0],
        observed_small_count,
        3,
    )

    assert positions_small_xz.shape == (
        expected_shape[0],
        observed_small_count,
        2,
    )

    return {
        "case_id": case_id,
        "compact_path": compact_path,
        "compact_manifest_path":
            compact_manifest_path,
        "compact_manifest":
            compact_manifest,
        "compact_keys": compact_keys,
        "positions_xyz": positions_xyz,
        "positions_small_xyz":
            positions_small_xyz,
        "positions_small_xz":
            positions_small_xz,
        "snapshot_times": snapshot_times,
        "source_snapshot_numbers":
            source_snapshot_numbers,
        "cycle_labels": cycle_labels,
        "particle_radii": particle_radii,
        "species_index": species_index,
        "small_mask": small_mask,
        "large_mask": large_mask,
        "small_radii": small_radii,
        "small_particle_volumes": (
            (4.0 / 3.0)
            * np.pi
            * small_radii**3
        ),
        "small_radius": small_radius,
        "large_radius": large_radius,
        "small_diameter": small_diameter,
        "large_diameter": large_diameter,
        "observed_size_ratio":
            observed_size_ratio,
        "lucy_support_to_small_diameter":
            lucy_support_to_small_diameter,
        "cycle_boundary_time":
            cycle_boundary_time,
        "training_indices":
            training_indices,
        "validation_indices":
            validation_indices,
        "excluded_indices":
            excluded_indices,
    }


N19_PILOT_CASE_ID = (
    "ratio_1_000000_num_1"
)

n19_pilot_row = (
    N19_CASE_PLAN.loc[
        N19_CASE_PLAN["case_id"].eq(
            N19_PILOT_CASE_ID
        )
    ]
    .iloc[0]
)

N19_PILOT_CASE = (
    n19_load_compact_case(
        n19_pilot_row
    )
)


print(
    "Loaded pilot case:",
    N19_PILOT_CASE["case_id"],
)

print(
    "Compact keys:",
    N19_PILOT_CASE["compact_keys"],
)

print(
    "All-particle XYZ shape:",
    N19_PILOT_CASE[
        "positions_xyz"
    ].shape,
)

print(
    "Small-particle XYZ shape:",
    N19_PILOT_CASE[
        "positions_small_xyz"
    ].shape,
)

print(
    "Observable-matched small-particle XZ shape:",
    N19_PILOT_CASE[
        "positions_small_xz"
    ].shape,
)

print(
    "Species labels:",
    np.unique(
        N19_PILOT_CASE[
            "species_index"
        ]
    ).tolist(),
)

print(
    "Small/large particle counts:",
    int(
        N19_PILOT_CASE[
            "small_mask"
        ].sum()
    ),
    int(
        N19_PILOT_CASE[
            "large_mask"
        ].sum()
    ),
)

print(
    "Small/large radii:",
    N19_PILOT_CASE["small_radius"],
    N19_PILOT_CASE["large_radius"],
)

print(
    "Small/large diameters:",
    N19_PILOT_CASE["small_diameter"],
    N19_PILOT_CASE["large_diameter"],
)

print(
    "Observed size ratio:",
    N19_PILOT_CASE[
        "observed_size_ratio"
    ],
)

print(
    "Lucy support / small-particle diameter:",
    N19_PILOT_CASE[
        "lucy_support_to_small_diameter"
    ],
)

print(
    "Training/validation snapshots:",
    len(
        N19_PILOT_CASE[
            "training_indices"
        ]
    ),
    len(
        N19_PILOT_CASE[
            "validation_indices"
        ]
    ),
)

print(
    "Excluded snapshot indices:",
    N19_PILOT_CASE[
        "excluded_indices"
    ].tolist(),
)

print(
    "Excluded times:",
    N19_PILOT_CASE[
        "snapshot_times"
    ][
        N19_PILOT_CASE[
            "excluded_indices"
        ]
    ].tolist(),
)

print(
    "Numerical particle arrays loaded:",
    True,
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "Files written:",
    False,
)

print(
    "Notebook 19 pilot-case loading passed:",
    True,
)

Loaded pilot case: ratio_1_000000_num_1
Compact keys: ['cycle_labels', 'particle_radii', 'positions_xyz', 'snapshot_times', 'source_byte_offsets', 'source_snapshot_numbers', 'species_index']
All-particle XYZ shape: (201, 10792, 3)
Small-particle XYZ shape: (201, 5396, 3)
Observable-matched small-particle XZ shape: (201, 5396, 2)
Species labels: [0, 1]
Small/large particle counts: 5396 5396
Small/large radii: 0.499614597 0.499614597
Small/large diameters: 0.999229194 0.999229194
Observed size ratio: 1.0
Lucy support / small-particle diameter: 3.0023142018006332
Training/validation snapshots: 101 99
Excluded snapshot indices: [200]
Excluded times: [1720.721256]
Numerical particle arrays loaded: True
Raw DEM files accessed: False
Files written: False
Notebook 19 pilot-case loading passed: True


## Observable-matched coarse-graining operator

The coarse-grained observable is the small-particle volume fraction on the radial plane. For a particle configuration $X$, define

$$
C_h(X)
=
\sum_{j:s_j=1}
\frac{V_j}{L_y}
W_h\!\left(\boldsymbol{x}-\boldsymbol{x}_j\right),
$$

where $s_j=1$ identifies the small species, $V_j$ is the particle volume, $L_y=10$ is the axial averaging length, and $W_h$ is the two-dimensional Lucy kernel with cutoff $h=3$.

Because the observable depends only on small-particle $(x,z)$ coordinates, the operator is also evaluated directly on

$$
X_{s,XZ}
\in
\mathbb{R}^{N_s\times 2}.
$$

The direct small-particle implementation is compared with the existing masked implementation on representative pilot snapshots. This verifies that the observable-matched route changes the particle representation, not the physical coarse-graining definition.

In [4]:
from scipy.spatial import cKDTree


# Reconstruct the exact reference grid used by the frozen
# Notebook 12/13 coarse-graining pipeline.
N19_REPOSITORY_ROOT = next(
    candidate
    for candidate in [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
    ]
    if (
        candidate
        / "results"
        / "tables"
        / "multi_parameter_route_order"
    ).is_dir()
)

N19_REFERENCE_CG_PATH = (
    N19_REPOSITORY_ROOT
    / "data"
    / "raw"
    / "CG_field_phi_small"
    / (
        "rotating_drum_bidisperse_size_ratio_"
        "1_000000_num_1.stat"
    )
)

assert N19_REFERENCE_CG_PATH.is_file()

n19_reference_data = np.loadtxt(
    N19_REFERENCE_CG_PATH,
    skiprows=2,
)

n19_reference_times = np.unique(
    n19_reference_data[:, 0]
)

N19_REFERENCE_X = np.unique(
    n19_reference_data[:, 1]
)

N19_REFERENCE_Z = np.unique(
    n19_reference_data[:, 2]
)

assert len(n19_reference_times) == 1
assert len(N19_REFERENCE_X) == 100
assert len(N19_REFERENCE_Z) == 100
assert len(n19_reference_data) == 10000

N19_REFERENCE_TIME = float(
    n19_reference_times[0]
)

N19_GRID_SPACING_X = float(
    np.median(
        np.diff(N19_REFERENCE_X)
    )
)

N19_GRID_SPACING_Z = float(
    np.median(
        np.diff(N19_REFERENCE_Z)
    )
)

n19_grid_x, n19_grid_z = np.meshgrid(
    N19_REFERENCE_X,
    N19_REFERENCE_Z,
    indexing="ij",
)

N19_GRID_POINTS = np.column_stack(
    [
        n19_grid_x.ravel(),
        n19_grid_z.ravel(),
    ]
)

N19_GRID_TREE = cKDTree(
    N19_GRID_POINTS
)

N19_LOCAL_LUCY_CUTOFF = 3.0
N19_LOCAL_AXIAL_LENGTH = 10.0


def n19_lucy_kernel_2d(
    distances,
    cutoff,
):
    distances = np.asarray(
        distances,
        dtype=np.float64,
    )

    q = distances / cutoff

    values = np.zeros_like(
        q,
        dtype=np.float64,
    )

    inside = q < 1.0
    q_inside = q[inside]

    values[inside] = (
        5.0
        / (
            np.pi
            * cutoff**2
        )
        * (
            1.0
            - 6.0 * q_inside**2
            + 8.0 * q_inside**3
            - 3.0 * q_inside**4
        )
    )

    return values


def n19_coarse_grain_small_only_xz(
    small_particle_xz,
    small_particle_volumes,
):
    """Coarse-grain a small-particle-only XZ state."""
    small_particle_xz = np.asarray(
        small_particle_xz,
        dtype=np.float64,
    )

    small_particle_volumes = np.asarray(
        small_particle_volumes,
        dtype=np.float64,
    )

    assert small_particle_xz.ndim == 2
    assert small_particle_xz.shape[1] == 2
    assert len(
        small_particle_xz
    ) == len(small_particle_volumes)

    assert np.isfinite(
        small_particle_xz
    ).all()

    assert np.isfinite(
        small_particle_volumes
    ).all()

    particle_tree = cKDTree(
        small_particle_xz
    )

    neighbour_map = (
        N19_GRID_TREE.sparse_distance_matrix(
            particle_tree,
            max_distance=N19_LOCAL_LUCY_CUTOFF,
            output_type="coo_matrix",
        )
    )

    kernel_values = n19_lucy_kernel_2d(
        neighbour_map.data,
        N19_LOCAL_LUCY_CUTOFF,
    )

    field_vector = np.bincount(
        neighbour_map.row,
        weights=(
            small_particle_volumes[
                neighbour_map.col
            ]
            * kernel_values
            / N19_LOCAL_AXIAL_LENGTH
        ),
        minlength=len(N19_GRID_POINTS),
    )

    return field_vector.reshape(
        len(N19_REFERENCE_X),
        len(N19_REFERENCE_Z),
    )


def n19_coarse_grain_masked_xz(
    all_particle_xz,
    small_mask,
    small_particle_volumes,
):
    """Existing masked implementation for regression comparison."""
    all_particle_xz = np.asarray(
        all_particle_xz,
        dtype=np.float64,
    )

    assert all_particle_xz.ndim == 2
    assert all_particle_xz.shape[1] == 2
    assert len(
        all_particle_xz
    ) == len(small_mask)

    return n19_coarse_grain_small_only_xz(
        all_particle_xz[
            small_mask
        ],
        small_particle_volumes,
    )


n19_pilot_small_volumes = (
    N19_PILOT_CASE[
        "small_particle_volumes"
    ]
)

n19_pilot_small_mask = (
    N19_PILOT_CASE[
        "small_mask"
    ]
)

n19_regression_indices = np.array(
    [
        N19_PILOT_CASE[
            "training_indices"
        ][0],
        N19_PILOT_CASE[
            "training_indices"
        ][-1],
        N19_PILOT_CASE[
            "validation_indices"
        ][-1],
    ],
    dtype=int,
)

n19_operator_regression_rows = []

for n19_index in n19_regression_indices:

    n19_all_particle_xz = (
        N19_PILOT_CASE["positions_xyz"][
            n19_index
        ][:, [0, 2]]
    )

    n19_small_particle_xz = (
        N19_PILOT_CASE[
            "positions_small_xz"
        ][
            n19_index
        ]
    )

    n19_masked_field = (
        n19_coarse_grain_masked_xz(
            n19_all_particle_xz,
            n19_pilot_small_mask,
            n19_pilot_small_volumes,
        )
    )

    n19_direct_field = (
        n19_coarse_grain_small_only_xz(
            n19_small_particle_xz,
            n19_pilot_small_volumes,
        )
    )

    n19_operator_regression_rows.append(
        {
            "snapshot_index": int(
                n19_index
            ),
            "snapshot_time": float(
                N19_PILOT_CASE[
                    "snapshot_times"
                ][n19_index]
            ),
            "maximum_absolute_difference": float(
                np.max(
                    np.abs(
                        n19_masked_field
                        - n19_direct_field
                    )
                )
            ),
            "relative_l2_difference": float(
                np.linalg.norm(
                    n19_masked_field
                    - n19_direct_field
                )
                / np.linalg.norm(
                    n19_masked_field
                )
            ),
            "minimum_field_value": float(
                n19_direct_field.min()
            ),
            "maximum_field_value": float(
                n19_direct_field.max()
            ),
        }
    )

N19_OPERATOR_REGRESSION = pd.DataFrame(
    n19_operator_regression_rows
)

display(
    N19_OPERATOR_REGRESSION.style.format(
        {
            "snapshot_time": "{:.6f}",
            "maximum_absolute_difference": "{:.3e}",
            "relative_l2_difference": "{:.3e}",
            "minimum_field_value": "{:.6e}",
            "maximum_field_value": "{:.6e}",
        }
    )
)

N19_MAX_OPERATOR_DIFFERENCE = float(
    N19_OPERATOR_REGRESSION[
        "maximum_absolute_difference"
    ].max()
)

N19_MAX_OPERATOR_RELATIVE_DIFFERENCE = float(
    N19_OPERATOR_REGRESSION[
        "relative_l2_difference"
    ].max()
)

print(
    "Reference grid:",
    (
        len(N19_REFERENCE_X),
        len(N19_REFERENCE_Z),
    ),
)

print(
    "Reference time:",
    N19_REFERENCE_TIME,
)

print(
    "Grid spacing:",
    N19_GRID_SPACING_X,
    N19_GRID_SPACING_Z,
)

print(
    "Lucy cutoff:",
    N19_LOCAL_LUCY_CUTOFF,
)

print(
    "Axial averaging length:",
    N19_LOCAL_AXIAL_LENGTH,
)

print(
    "Maximum masked/direct absolute difference:",
    N19_MAX_OPERATOR_DIFFERENCE,
)

print(
    "Maximum masked/direct relative difference:",
    N19_MAX_OPERATOR_RELATIVE_DIFFERENCE,
)

print(
    "Small-only and masked coarse-graining agree:",
    N19_MAX_OPERATOR_DIFFERENCE < 1.0e-14,
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "Files written:",
    False,
)

assert np.isclose(
    N19_REFERENCE_TIME,
    N19_COMPARISON_TIME_MAX,
    rtol=0.0,
    atol=1.0e-12,
)

assert N19_MAX_OPERATOR_DIFFERENCE < 1.0e-14
assert N19_MAX_OPERATOR_RELATIVE_DIFFERENCE < 1.0e-14

print(
    "Notebook 19 coarse-graining operator regression passed:",
    True,
)

,snapshot_index,snapshot_time,maximum_absolute_difference,relative_l2_difference,minimum_field_value,maximum_field_value
0,0,1378.133363,0.000e+00,0.000e+00,0.000000e+00,3.510258e-01
1,100,1550.184969,0.000e+00,0.000e+00,0.000000e+00,3.525091e-01
2,199,1720.516059,0.000e+00,0.000e+00,0.000000e+00,3.743681e-01


Reference grid: (100, 100)
Reference time: 1720.72
Grid spacing: 0.6600000000000001 0.6600000000000001
Lucy cutoff: 3.0
Axial averaging length: 10.0
Maximum masked/direct absolute difference: 0.0
Maximum masked/direct relative difference: 0.0
Small-only and masked coarse-graining agree: True
Raw DEM files accessed: False
Files written: False
Notebook 19 coarse-graining operator regression passed: True


## Observable-matched small-species POD

The new representation applies POD only to the coordinates that can influence the measured observable,

$$
X_{s,XZ,t}
\in
\mathbb{R}^{2N_s},
$$

rather than to all-particle $XYZ$ coordinates.

The training matrix is

$$
S_{\mathrm{train}}
=
\begin{bmatrix}
\operatorname{vec}(X_{s,XZ,t_1}) &
\cdots &
\operatorname{vec}(X_{s,XZ,t_{101}})
\end{bmatrix},
$$

with training-mean centring,

$$
S_{\mathrm{train,c}}
=
S_{\mathrm{train}}
-
\overline{S}_{\mathrm{train}}.
$$

The validation particle states are projected onto the frozen training POD basis. Their reconstructed small-particle coordinates are then coarse-grained directly using the validated operator from the previous cell.

This isolates the effect of removing particle coordinates that are irrelevant to the small-species volume-fraction observable.

In [5]:
def n19_calculate_thin_pod(
    matrix,
):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    assert matrix.ndim == 2
    assert np.isfinite(matrix).all()

    (
        left_vectors,
        singular_values,
        right_vectors_transpose,
    ) = np.linalg.svd(
        matrix,
        full_matrices=False,
    )

    squared_singular_values = (
        singular_values**2
    )

    cumulative_energy = (
        np.cumsum(
            squared_singular_values
        )
        / np.sum(
            squared_singular_values
        )
    )

    cumulative_energy[-1] = 1.0

    numerical_tolerance = (
        np.finfo(
            singular_values.dtype
        ).eps
        * max(matrix.shape)
        * singular_values[0]
    )

    numerical_rank = int(
        np.sum(
            singular_values
            > numerical_tolerance
        )
    )

    return (
        left_vectors,
        singular_values,
        right_vectors_transpose,
        cumulative_energy,
        numerical_rank,
    )


def n19_coarse_grain_small_xz_sequence(
    small_xz_sequence,
    small_particle_volumes,
    label=None,
):
    small_xz_sequence = np.asarray(
        small_xz_sequence,
        dtype=np.float64,
    )

    assert small_xz_sequence.ndim == 3
    assert small_xz_sequence.shape[2] == 2
    assert np.isfinite(
        small_xz_sequence
    ).all()

    fields = np.empty(
        (
            len(small_xz_sequence),
            len(N19_REFERENCE_X),
            len(N19_REFERENCE_Z),
        ),
        dtype=np.float64,
    )

    for local_index, state in enumerate(
        small_xz_sequence
    ):
        fields[local_index] = (
            n19_coarse_grain_small_only_xz(
                state,
                small_particle_volumes,
            )
        )

        completed = local_index + 1

        if (
            label is not None
            and (
                completed % 25 == 0
                or completed == len(
                    small_xz_sequence
                )
            )
        ):
            print(
                label,
                completed,
                "of",
                len(small_xz_sequence),
            )

    return fields


def n19_field_summary_metrics(
    prediction_fields,
    truth_fields,
    rank,
):
    prediction_matrix = (
        prediction_fields.reshape(
            len(prediction_fields),
            -1,
        )
    )

    truth_matrix = (
        truth_fields.reshape(
            len(truth_fields),
            -1,
        )
    )

    difference_matrix = (
        prediction_matrix
        - truth_matrix
    )

    truth_norms = np.linalg.norm(
        truth_matrix,
        axis=1,
    )

    per_snapshot_errors = (
        100.0
        * np.linalg.norm(
            difference_matrix,
            axis=1,
        )
        / truth_norms
    )

    prediction_average = (
        prediction_fields.mean(axis=0)
    )

    truth_average = (
        truth_fields.mean(axis=0)
    )

    return {
        "rank": int(rank),
        "spacetime_relative_l2_error_percent": (
            100.0
            * np.linalg.norm(
                difference_matrix
            )
            / np.linalg.norm(
                truth_matrix
            )
        ),
        "mean_snapshot_l2_error_percent": (
            float(
                per_snapshot_errors.mean()
            )
        ),
        "median_snapshot_l2_error_percent": (
            float(
                np.median(
                    per_snapshot_errors
                )
            )
        ),
        "p95_snapshot_l2_error_percent": (
            float(
                np.percentile(
                    per_snapshot_errors,
                    95.0,
                )
            )
        ),
        "maximum_snapshot_l2_error_percent": (
            float(
                per_snapshot_errors.max()
            )
        ),
        "time_averaged_l2_error_percent": (
            100.0
            * np.linalg.norm(
                prediction_average
                - truth_average
            )
            / np.linalg.norm(
                truth_average
            )
        ),
        "minimum_volume_fraction": float(
            prediction_matrix.min()
        ),
        "maximum_volume_fraction": float(
            prediction_matrix.max()
        ),
        "grid_fraction_phi_lt_zero_percent": (
            100.0
            * np.mean(
                prediction_matrix < 0.0
            )
        ),
    }


n19_small_training_indices = (
    N19_PILOT_CASE[
        "training_indices"
    ]
)

n19_small_validation_indices = (
    N19_PILOT_CASE[
        "validation_indices"
    ]
)

n19_small_training_matrix = (
    N19_PILOT_CASE[
        "positions_small_xz"
    ][
        n19_small_training_indices
    ]
    .reshape(
        len(n19_small_training_indices),
        -1,
    )
    .T
)

n19_small_validation_matrix = (
    N19_PILOT_CASE[
        "positions_small_xz"
    ][
        n19_small_validation_indices
    ]
    .reshape(
        len(n19_small_validation_indices),
        -1,
    )
    .T
)

n19_small_training_mean = (
    n19_small_training_matrix.mean(
        axis=1,
        keepdims=True,
    )
)

n19_small_training_centered = (
    n19_small_training_matrix
    - n19_small_training_mean
)

n19_small_validation_centered = (
    n19_small_validation_matrix
    - n19_small_training_mean
)

(
    N19_SMALL_POD_MODES,
    N19_SMALL_POD_SINGULAR_VALUES,
    N19_SMALL_POD_RIGHT_SINGULAR_VECTORS,
    N19_SMALL_POD_CUMULATIVE_ENERGY,
    N19_SMALL_POD_NUMERICAL_RANK,
) = n19_calculate_thin_pod(
    n19_small_training_centered
)

assert n19_small_training_matrix.shape == (
    2
    * int(
        N19_PILOT_CASE[
            "small_mask"
        ].sum()
    ),
    len(n19_small_training_indices),
)

assert n19_small_validation_matrix.shape == (
    2
    * int(
        N19_PILOT_CASE[
            "small_mask"
        ].sum()
    ),
    len(n19_small_validation_indices),
)

assert (
    N19_SMALL_POD_NUMERICAL_RANK
    >= 100
)

assert np.allclose(
    N19_SMALL_POD_MODES.T
    @ N19_SMALL_POD_MODES,
    np.eye(
        N19_SMALL_POD_MODES.shape[1]
    ),
    rtol=0.0,
    atol=1.0e-12,
)

assert np.isclose(
    n19_small_training_centered.mean(
        axis=1
    ).max(),
    0.0,
    rtol=0.0,
    atol=1.0e-12,
)

print(
    "Small-only training matrix:",
    n19_small_training_matrix.shape,
)

print(
    "Small-only validation matrix:",
    n19_small_validation_matrix.shape,
)

print(
    "Small-only POD numerical rank:",
    N19_SMALL_POD_NUMERICAL_RANK,
)

print(
    "Small-only POD orthonormality residual:",
    float(
        np.max(
            np.abs(
                N19_SMALL_POD_MODES.T
                @ N19_SMALL_POD_MODES
                - np.eye(
                    N19_SMALL_POD_MODES.shape[1]
                )
            )
        )
    ),
)

print(
    "Small-only cumulative energy:",
    {
        rank: float(
            N19_SMALL_POD_CUMULATIVE_ENERGY[
                rank - 1
            ]
        )
        for rank in (8, 14, 27, 100)
    },
)

print(
    "Building true validation fields:"
)

N19_SMALL_ONLY_TRUE_VALIDATION_FIELDS = (
    n19_coarse_grain_small_xz_sequence(
        N19_PILOT_CASE[
            "positions_small_xz"
        ][
            n19_small_validation_indices
        ],
        N19_PILOT_CASE[
            "small_particle_volumes"
        ],
        label="True validation fields:",
    )
)

N19_SMALL_ONLY_PILOT_RESULTS = []

for n19_rank in (
    0,
    8,
    14,
    27,
    100,
):
    if n19_rank == 0:
        n19_reconstructed_matrix = (
            np.repeat(
                n19_small_training_mean,
                len(
                    n19_small_validation_indices
                ),
                axis=1,
            )
        )
    else:
        n19_basis = (
            N19_SMALL_POD_MODES[
                :,
                :n19_rank,
            ]
        )

        n19_validation_coefficients = (
            n19_basis.T
            @ n19_small_validation_centered
        )

        n19_reconstructed_matrix = (
            n19_small_training_mean
            + n19_basis
            @ n19_validation_coefficients
        )

    n19_reconstructed_small_xz = (
        n19_reconstructed_matrix.T.reshape(
            len(n19_small_validation_indices),
            int(
                N19_PILOT_CASE[
                    "small_mask"
                ].sum()
            ),
            2,
        )
    )

    print(
        f"Building reconstructed rank-{n19_rank} "
        "validation fields:"
    )

    n19_reconstructed_fields = (
        n19_coarse_grain_small_xz_sequence(
            n19_reconstructed_small_xz,
            N19_PILOT_CASE[
                "small_particle_volumes"
            ],
            label=(
                f"Rank {n19_rank} fields:"
            ),
        )
    )

    n19_metrics = (
        n19_field_summary_metrics(
            n19_reconstructed_fields,
            N19_SMALL_ONLY_TRUE_VALIDATION_FIELDS,
            n19_rank,
        )
    )

    n19_metrics[
        "training_energy_percent"
    ] = (
        0.0
        if n19_rank == 0
        else 100.0
        * N19_SMALL_POD_CUMULATIVE_ENERGY[
            n19_rank - 1
        ]
    )

    N19_SMALL_ONLY_PILOT_RESULTS.append(
        n19_metrics
    )

N19_SMALL_ONLY_PILOT_RESULTS = pd.DataFrame(
    N19_SMALL_ONLY_PILOT_RESULTS
).sort_values(
    "rank"
).reset_index(
    drop=True
)

display(
    N19_SMALL_ONLY_PILOT_RESULTS.style.format(
        {
            column: "{:.6f}"
            for column
            in N19_SMALL_ONLY_PILOT_RESULTS.columns
            if column != "rank"
        }
    )
)

assert len(
    N19_SMALL_ONLY_PILOT_RESULTS
) == 5

assert np.isfinite(
    N19_SMALL_ONLY_PILOT_RESULTS.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print(
    "Observable-matched small-only POD completed:",
    True,
)

print(
    "Validation fields computed:",
    len(
        N19_SMALL_ONLY_TRUE_VALIDATION_FIELDS
    ),
)

print(
    "Files written:",
    False,
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "Notebook 19 small-only pilot POD passed:",
    True,
)

Small-only training matrix: (10792, 101)
Small-only validation matrix: (10792, 99)
Small-only POD numerical rank: 100
Small-only POD orthonormality residual: 2.3314683517128287e-15
Small-only cumulative energy: {8: 0.9600016275553895, 14: 0.9912142020842155, 27: 0.9994233432715137, 100: 1.0000000000000002}
Building true validation fields:
True validation fields: 25 of 99
True validation fields: 50 of 99
True validation fields: 75 of 99
True validation fields: 99 of 99
Building reconstructed rank-0 validation fields:
Rank 0 fields: 25 of 99
Rank 0 fields: 50 of 99
Rank 0 fields: 75 of 99
Rank 0 fields: 99 of 99
Building reconstructed rank-8 validation fields:
Rank 8 fields: 25 of 99
Rank 8 fields: 50 of 99
Rank 8 fields: 75 of 99
Rank 8 fields: 99 of 99
Building reconstructed rank-14 validation fields:
Rank 14 fields: 25 of 99
Rank 14 fields: 50 of 99
Rank 14 fields: 75 of 99
Rank 14 fields: 99 of 99
Building reconstructed rank-27 validation fields:
Rank 27 fields: 25 of 99
Rank 27 fiel

,rank,spacetime_relative_l2_error_percent,mean_snapshot_l2_error_percent,median_snapshot_l2_error_percent,p95_snapshot_l2_error_percent,maximum_snapshot_l2_error_percent,time_averaged_l2_error_percent,minimum_volume_fraction,maximum_volume_fraction,grid_fraction_phi_lt_zero_percent,training_energy_percent
0,0,577.294936,577.298469,576.860821,579.978044,580.318551,580.725546,0.000000,17.231289,0.000000,0.000000
1,8,103.364067,98.125608,79.926170,143.606845,145.932827,84.921304,0.000000,2.025867,0.000000,96.000163
2,14,101.160475,95.719712,81.407756,144.571084,145.344037,80.639671,0.000000,2.372336,0.000000,99.121420
3,27,98.911327,92.948121,80.255520,143.826120,144.838534,78.512980,0.000000,2.266410,0.000000,99.942334
4,100,93.301591,87.088305,71.588492,137.223257,138.674295,74.816481,0.000000,2.042362,0.000000,100.000000


Observable-matched small-only POD completed: True
Validation fields computed: 99
Files written: False
Raw DEM files accessed: False
Notebook 19 small-only pilot POD passed: True


## Small-only versus all-particle $XZ$ comparison

The small-only POD is now compared with the previously saved all-particle $XZ$ baseline from Notebook 13.

The comparison is made at the same ranks and on the same pilot validation snapshots. Rank zero should agree to numerical precision because both routes use the same training-mean small-particle configuration before coarse-graining.

For positive ranks, any difference measures the effect of removing the large-particle coordinates from the POD state.

In [6]:
N19_SAVED_XZ_FIELD_PATH = (
    N19_REPOSITORY_ROOT
    / "results"
    / "tables"
    / "xz_particle_representation_sensitivity"
    / (
        "xz_particle_first_field_"
        "ratio_1_000000_num_1.csv"
    )
)

assert N19_SAVED_XZ_FIELD_PATH.is_file()

N19_SAVED_XZ_FIELD = pd.read_csv(
    N19_SAVED_XZ_FIELD_PATH
)

assert set(
    [
        "rank",
        "spacetime_relative_l2_error_percent",
        "time_averaged_l2_error_percent",
    ]
).issubset(
    N19_SAVED_XZ_FIELD.columns
)

N19_SAVED_XZ_FIELD = (
    N19_SAVED_XZ_FIELD.loc[
        N19_SAVED_XZ_FIELD[
            "rank"
        ].isin(
            [
                0,
                8,
                14,
                27,
                100,
            ]
        )
    ]
    .sort_values("rank")
    .reset_index(drop=True)
)

N19_SMALL_ONLY_COMPARISON = (
    N19_SMALL_ONLY_PILOT_RESULTS[
        [
            "rank",
            "spacetime_relative_l2_error_percent",
            "time_averaged_l2_error_percent",
        ]
    ]
    .rename(
        columns={
            "spacetime_relative_l2_error_percent":
                "small_only_spacetime_error_percent",
            "time_averaged_l2_error_percent":
                "small_only_time_averaged_error_percent",
        }
    )
    .merge(
        N19_SAVED_XZ_FIELD[
            [
                "rank",
                "spacetime_relative_l2_error_percent",
                "time_averaged_l2_error_percent",
            ]
        ].rename(
            columns={
                "spacetime_relative_l2_error_percent":
                    "all_particle_xz_spacetime_error_percent",
                "time_averaged_l2_error_percent":
                    "all_particle_xz_time_averaged_error_percent",
            }
        ),
        on="rank",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("rank")
    .reset_index(drop=True)
)

N19_SMALL_ONLY_COMPARISON[
    "spacetime_error_difference_percent"
] = (
    N19_SMALL_ONLY_COMPARISON[
        "small_only_spacetime_error_percent"
    ]
    - N19_SMALL_ONLY_COMPARISON[
        "all_particle_xz_spacetime_error_percent"
    ]
)

N19_SMALL_ONLY_COMPARISON[
    "time_averaged_error_difference_percent"
] = (
    N19_SMALL_ONLY_COMPARISON[
        "small_only_time_averaged_error_percent"
    ]
    - N19_SMALL_ONLY_COMPARISON[
        "all_particle_xz_time_averaged_error_percent"
    ]
)

N19_SMALL_ONLY_COMPARISON[
    "spacetime_error_ratio"
] = (
    N19_SMALL_ONLY_COMPARISON[
        "small_only_spacetime_error_percent"
    ]
    / N19_SMALL_ONLY_COMPARISON[
        "all_particle_xz_spacetime_error_percent"
    ]
)

display(
    N19_SMALL_ONLY_COMPARISON.style.format(
        {
            column: "{:.6f}"
            for column
            in N19_SMALL_ONLY_COMPARISON.columns
            if column != "rank"
        }
    )
)

n19_rank_zero_row = (
    N19_SMALL_ONLY_COMPARISON.loc[
        N19_SMALL_ONLY_COMPARISON[
            "rank"
        ].eq(0)
    ]
    .iloc[0]
)

n19_rank_zero_difference = max(
    abs(
        float(
            n19_rank_zero_row[
                "spacetime_error_difference_percent"
            ]
        )
    ),
    abs(
        float(
            n19_rank_zero_row[
                "time_averaged_error_difference_percent"
            ]
        )
    ),
)

n19_rank_100_row = (
    N19_SMALL_ONLY_COMPARISON.loc[
        N19_SMALL_ONLY_COMPARISON[
            "rank"
        ].eq(100)
    ]
    .iloc[0]
)

print(
    "Saved all-particle XZ rows:",
    len(N19_SAVED_XZ_FIELD),
)

print(
    "Comparison ranks:",
    N19_SMALL_ONLY_COMPARISON[
        "rank"
    ].tolist(),
)

print(
    "Maximum rank-zero metric difference:",
    n19_rank_zero_difference,
)

print(
    "Rank-100 all-particle XZ error (%):",
    float(
        n19_rank_100_row[
            "all_particle_xz_spacetime_error_percent"
        ]
    ),
)

print(
    "Rank-100 small-only XZ error (%):",
    float(
        n19_rank_100_row[
            "small_only_spacetime_error_percent"
        ]
    ),
)

print(
    "Rank-100 small-only minus all-particle "
    "XZ error (percentage points):",
    float(
        n19_rank_100_row[
            "spacetime_error_difference_percent"
        ]
    ),
)

print(
    "Files written:",
    False,
)

print(
    "Raw DEM files accessed:",
    False,
)

assert len(
    N19_SMALL_ONLY_COMPARISON
) == 5

assert n19_rank_zero_difference < 1.0e-10

assert np.isfinite(
    N19_SMALL_ONLY_COMPARISON.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print(
    "Small-only versus all-particle XZ comparison passed:",
    True,
)

,rank,small_only_spacetime_error_percent,small_only_time_averaged_error_percent,all_particle_xz_spacetime_error_percent,all_particle_xz_time_averaged_error_percent,spacetime_error_difference_percent,time_averaged_error_difference_percent,spacetime_error_ratio
0,0,577.294936,580.725546,577.294936,580.725546,0.000000,0.000000,1.000000
1,8,103.364067,84.921304,100.842931,82.336478,2.521136,2.584826,1.025001
2,14,101.160475,80.639671,99.112102,78.375757,2.048373,2.263913,1.020667
3,27,98.911327,78.512980,97.015032,76.567254,1.896295,1.945726,1.019546
4,100,93.301591,74.816481,92.687320,73.646940,0.614271,1.169540,1.006627


Saved all-particle XZ rows: 5
Comparison ranks: [0, 8, 14, 27, 100]
Maximum rank-zero metric difference: 0.0
Rank-100 all-particle XZ error (%): 92.6873198150537
Rank-100 small-only XZ error (%): 93.30159094696675
Rank-100 small-only minus all-particle XZ error (percentage points): 0.6142711319130569
Files written: False
Raw DEM files accessed: False
Small-only versus all-particle XZ comparison passed: True


## Quantified particle-row continuity audit

The compact caches contain labelled particle rows but do not provide persistent particle identifiers. A continuity audit is therefore used to quantify how consistently each stored row behaves between adjacent snapshots.

For each species and each consecutive pair of snapshots, the following are recorded:

- the displacement of each stored row;
- the distance to the nearest same-species particle in the next snapshot;
- the fraction of rows for which the stored partner is the nearest same-species particle;
- robust summaries of row displacement and row-to-nearest-neighbour ratios.

This is a diagnostic of ordering continuity, not proof of particle identity. It is used to determine whether the particle-row representation is sufficiently coherent for a labelled-coordinate POD.

In [7]:
from scipy.spatial import cKDTree


def n19_continuity_summary_for_case(
    case_row,
):
    case_data = n19_load_compact_case(
        case_row
    )

    positions_xyz = (
        case_data["positions_xyz"]
    )

    snapshot_times = (
        case_data["snapshot_times"]
    )

    species_index = (
        case_data["species_index"]
    )

    continuity_rows = []

    for species_label in np.unique(
        species_index
    ):
        species_mask = (
            species_index
            == species_label
        )

        species_positions = (
            positions_xyz[
                :,
                species_mask,
                :,
            ]
        )

        interval_records = []

        for snapshot_index in range(
            len(snapshot_times) - 1
        ):
            current_positions = (
                species_positions[
                    snapshot_index
                ]
            )

            next_positions = (
                species_positions[
                    snapshot_index + 1
                ]
            )

            time_step = float(
                snapshot_times[
                    snapshot_index + 1
                ]
                - snapshot_times[
                    snapshot_index
                ]
            )

            row_displacements = np.linalg.norm(
                next_positions
                - current_positions,
                axis=1,
            )

            next_tree = cKDTree(
                next_positions
            )

            nearest_distances, nearest_indices = (
                next_tree.query(
                    current_positions,
                    k=1,
                )
            )

            stored_partner_is_nearest = (
                nearest_indices
                == np.arange(
                    len(current_positions)
                )
            )

            row_to_nearest_ratio = (
                row_displacements
                / np.maximum(
                    nearest_distances,
                    1.0e-14,
                )
            )

            interval_records.append(
                {
                    "case_id": str(
                        case_row["case_id"]
                    ),
                    "size_ratio": float(
                        case_row["size_ratio"]
                    ),
                    "species": int(
                        species_label
                    ),
                    "snapshot_index": int(
                        snapshot_index
                    ),
                    "time_step": time_step,
                    "row_displacement_median": float(
                        np.median(
                            row_displacements
                        )
                    ),
                    "row_displacement_p95": float(
                        np.percentile(
                            row_displacements,
                            95.0,
                        )
                    ),
                    "row_displacement_p99": float(
                        np.percentile(
                            row_displacements,
                            99.0,
                        )
                    ),
                    "row_displacement_maximum": float(
                        row_displacements.max()
                    ),
                    "nearest_distance_median": float(
                        np.median(
                            nearest_distances
                        )
                    ),
                    "nearest_distance_p95": float(
                        np.percentile(
                            nearest_distances,
                            95.0,
                        )
                    ),
                    "row_to_nearest_ratio_median": float(
                        np.median(
                            row_to_nearest_ratio
                        )
                    ),
                    "row_to_nearest_ratio_p95": float(
                        np.percentile(
                            row_to_nearest_ratio,
                            95.0,
                        )
                    ),
                    "stored_partner_nearest_fraction": float(
                        np.mean(
                            stored_partner_is_nearest
                        )
                    ),
                }
            )

        interval_table = pd.DataFrame(
            interval_records
        )

        continuity_rows.append(
            {
                "case_id": str(
                    case_row["case_id"]
                ),
                "size_ratio": float(
                    case_row["size_ratio"]
                ),
                "species": int(
                    species_label
                ),
                "particle_count": int(
                    species_mask.sum()
                ),
                "interval_count": len(
                    interval_table
                ),
                "median_row_displacement": float(
                    interval_table[
                        "row_displacement_median"
                    ].median()
                ),
                "p95_row_displacement": float(
                    interval_table[
                        "row_displacement_p95"
                    ].max()
                ),
                "p99_row_displacement": float(
                    interval_table[
                        "row_displacement_p99"
                    ].max()
                ),
                "maximum_row_displacement": float(
                    interval_table[
                        "row_displacement_maximum"
                    ].max()
                ),
                "median_nearest_distance": float(
                    interval_table[
                        "nearest_distance_median"
                    ].median()
                ),
                "p95_nearest_distance": float(
                    interval_table[
                        "nearest_distance_p95"
                    ].max()
                ),
                "median_row_to_nearest_ratio": float(
                    interval_table[
                        "row_to_nearest_ratio_median"
                    ].median()
                ),
                "p95_row_to_nearest_ratio": float(
                    interval_table[
                        "row_to_nearest_ratio_p95"
                    ].max()
                ),
                "mean_stored_partner_nearest_fraction": float(
                    interval_table[
                        "stored_partner_nearest_fraction"
                    ].mean()
                ),
                "minimum_stored_partner_nearest_fraction": float(
                    interval_table[
                        "stored_partner_nearest_fraction"
                    ].min()
                ),
            }
        )

    return pd.DataFrame(
        continuity_rows
    )


N19_CONTINUITY_SUMMARIES = []

for _, n19_case_row in (
    N19_CASE_PLAN.sort_values(
        "size_ratio"
    ).iterrows()
):
    print(
        "Auditing row continuity:",
        n19_case_row["case_id"],
    )

    n19_case_summary = (
        n19_continuity_summary_for_case(
            n19_case_row
        )
    )

    N19_CONTINUITY_SUMMARIES.append(
        n19_case_summary
    )

N19_CONTINUITY_SUMMARY = pd.concat(
    N19_CONTINUITY_SUMMARIES,
    ignore_index=True,
)

display(
    N19_CONTINUITY_SUMMARY.style.format(
        {
            "size_ratio": "{:.6f}",
            "median_row_displacement": "{:.6e}",
            "p95_row_displacement": "{:.6e}",
            "p99_row_displacement": "{:.6e}",
            "maximum_row_displacement": "{:.6e}",
            "median_nearest_distance": "{:.6e}",
            "p95_nearest_distance": "{:.6e}",
            "median_row_to_nearest_ratio": "{:.6f}",
            "p95_row_to_nearest_ratio": "{:.6f}",
            "mean_stored_partner_nearest_fraction": "{:.6f}",
            "minimum_stored_partner_nearest_fraction": "{:.6f}",
        }
    )
)

assert len(
    N19_CONTINUITY_SUMMARY
) == (
    2
    * len(N19_CASE_PLAN)
)

assert (
    N19_CONTINUITY_SUMMARY[
        "interval_count"
    ]
    == 200
).all()

assert np.isfinite(
    N19_CONTINUITY_SUMMARY.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print(
    "Cases audited:",
    N19_CONTINUITY_SUMMARY[
        "case_id"
    ].nunique(),
)

print(
    "Species audited:",
    sorted(
        N19_CONTINUITY_SUMMARY[
            "species"
        ].unique()
    ),
)

print(
    "Adjacent intervals per case:",
    sorted(
        N19_CONTINUITY_SUMMARY[
            "interval_count"
        ].unique()
    ),
)

print(
    "Mean stored-partner-nearest fraction:",
    float(
        N19_CONTINUITY_SUMMARY[
            "mean_stored_partner_nearest_fraction"
        ].mean()
    ),
)

print(
    "Minimum stored-partner-nearest fraction:",
    float(
        N19_CONTINUITY_SUMMARY[
            "minimum_stored_partner_nearest_fraction"
        ].min()
    ),
)

print(
    "Interpretation:",
    "descriptive row-continuity diagnostic; "
    "not proof of persistent particle identity.",
)

print(
    "Files written:",
    False,
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "Notebook 19 row-continuity audit passed:",
    True,
)

Auditing row continuity: ratio_1_000000_num_1
Auditing row continuity: ratio_1_252525_num_26
Auditing row continuity: ratio_1_494949_num_50
Auditing row continuity: ratio_1_747475_num_75
Auditing row continuity: ratio_2_000000_num_100


,case_id,size_ratio,species,particle_count,interval_count,median_row_displacement,p95_row_displacement,p99_row_displacement,maximum_row_displacement,median_nearest_distance,p95_nearest_distance,median_row_to_nearest_ratio,p95_row_to_nearest_ratio,mean_stored_partner_nearest_fraction,minimum_stored_partner_nearest_fraction
0,ratio_1_000000_num_1,1.000000,0,5396,200,1.575321e+00,6.518856e+00,9.238237e+00,1.241183e+01,6.429536e-01,1.101187e+00,2.503190,10.233677,0.057996,0.045589
1,ratio_1_000000_num_1,1.000000,1,5396,200,1.543190e+00,6.237070e+00,9.258979e+00,1.230686e+01,6.399962e-01,1.105830e+00,2.438253,10.006051,0.059126,0.045033
2,ratio_1_252525_num_26,1.252525,0,3925,200,1.649292e+00,6.934585e+00,9.451382e+00,1.246890e+01,6.733538e-01,1.167015e+00,2.632450,10.254163,0.048815,0.036433
3,ratio_1_252525_num_26,1.252525,1,7701,200,1.429202e+00,5.836295e+00,8.763980e+00,1.342309e+01,5.345988e-01,1.057892e+00,2.581455,10.928479,0.045288,0.032983
4,ratio_1_494949_num_50,1.494949,0,3137,200,1.681178e+00,7.100382e+00,9.398786e+00,1.290847e+01,6.791256e-01,1.214681e+00,2.682324,10.400371,0.039270,0.028371
5,ratio_1_494949_num_50,1.494949,1,10462,200,1.360369e+00,5.305821e+00,8.185774e+00,1.269634e+01,4.563447e-01,1.007652e+00,2.845463,11.697768,0.035108,0.023992
6,ratio_1_747475_num_75,1.747475,0,2623,200,1.696209e+00,7.177378e+00,9.423466e+00,1.192008e+01,6.947688e-01,1.258411e+00,2.662807,10.369864,0.040128,0.028974
7,ratio_1_747475_num_75,1.747475,1,13961,200,1.342738e+00,5.284191e+00,7.884660e+00,1.334138e+01,4.040338e-01,9.366932e-01,3.174904,13.053664,0.027807,0.018838
8,ratio_2_000000_num_100,2.000000,0,2280,200,1.702707e+00,7.094768e+00,9.440724e+00,1.210230e+01,7.105360e-01,1.294993e+00,2.630665,9.821121,0.041629,0.030263
9,ratio_2_000000_num_100,2.000000,1,18169,200,1.334441e+00,5.209183e+00,7.658349e+00,1.291980e+01,3.648411e-01,8.782052e-01,3.506908,14.303181,0.023448,0.014475


Cases audited: 5
Species audited: [0, 1]
Adjacent intervals per case: [200]
Mean stored-partner-nearest fraction: 0.041861618017221616
Minimum stored-partner-nearest fraction: 0.014475205019538775
Interpretation: descriptive row-continuity diagnostic; not proof of persistent particle identity.
Files written: False
Raw DEM files accessed: False
Notebook 19 row-continuity audit passed: True


In [8]:
def n19_run_small_only_case(
    case_row,
    ranks=(0, 8, 14, 27, 100),
    progress=True,
):
    case_data = n19_load_compact_case(case_row)

    case_id = str(case_row["case_id"])
    size_ratio = float(case_row["size_ratio"])

    positions_small_xz = case_data["positions_small_xz"]
    training_indices = case_data["training_indices"]
    validation_indices = case_data["validation_indices"]
    small_particle_volumes = (
        case_data["small_particle_volumes"]
    )

    number_of_small_particles = (
        positions_small_xz.shape[1]
    )

    training_matrix = (
        positions_small_xz[training_indices]
        .reshape(len(training_indices), -1)
        .T
    )

    validation_matrix = (
        positions_small_xz[validation_indices]
        .reshape(len(validation_indices), -1)
        .T
    )

    training_mean = training_matrix.mean(
        axis=1,
        keepdims=True,
    )

    training_centered = (
        training_matrix - training_mean
    )

    validation_centered = (
        validation_matrix - training_mean
    )

    (
        pod_modes,
        singular_values,
        right_singular_vectors_transpose,
        cumulative_energy,
        numerical_rank,
    ) = n19_calculate_thin_pod(
        training_centered
    )

    assert numerical_rank >= max(ranks)

    true_validation_fields = (
        n19_coarse_grain_small_xz_sequence(
            positions_small_xz[validation_indices],
            small_particle_volumes,
            label=(
                f"{case_id} true validation:"
                if progress
                else None
            ),
        )
    )

    result_rows = []

    for rank in ranks:
        if rank == 0:
            reconstructed_matrix = np.repeat(
                training_mean,
                len(validation_indices),
                axis=1,
            )
        else:
            basis = pod_modes[:, :rank]

            validation_coefficients = (
                basis.T @ validation_centered
            )

            reconstructed_matrix = (
                training_mean
                + basis @ validation_coefficients
            )

        reconstructed_small_xz = (
            reconstructed_matrix.T.reshape(
                len(validation_indices),
                number_of_small_particles,
                2,
            )
        )

        reconstructed_fields = (
            n19_coarse_grain_small_xz_sequence(
                reconstructed_small_xz,
                small_particle_volumes,
                label=(
                    f"{case_id} rank {rank}:"
                    if progress
                    else None
                ),
            )
        )

        metrics = n19_field_summary_metrics(
            reconstructed_fields,
            true_validation_fields,
            rank,
        )

        metrics.update(
            {
                "case_id": case_id,
                "size_ratio": size_ratio,
                "pod_numerical_rank": int(
                    numerical_rank
                ),
                "training_snapshot_count": int(
                    len(training_indices)
                ),
                "validation_snapshot_count": int(
                    len(validation_indices)
                ),
                "training_energy_percent": (
                    0.0
                    if rank == 0
                    else 100.0
                    * cumulative_energy[rank - 1]
                ),
            }
        )

        result_rows.append(metrics)

    return (
        pd.DataFrame(result_rows)
        .sort_values("rank")
        .reset_index(drop=True)
    )


N19_SMALL_ONLY_SWEEP_FRAMES = [
    N19_SMALL_ONLY_PILOT_RESULTS.assign(
        case_id=N19_PILOT_CASE_ID,
        size_ratio=1.0,
        pod_numerical_rank=(
            N19_SMALL_POD_NUMERICAL_RANK
        ),
        training_snapshot_count=101,
        validation_snapshot_count=99,
    )
]

for _, n19_case_row in (
    N19_CASE_PLAN.sort_values(
        "size_ratio"
    ).iterrows()
):
    if (
        str(n19_case_row["case_id"])
        == N19_PILOT_CASE_ID
    ):
        continue

    print(
        "\nRunning small-only case:",
        n19_case_row["case_id"],
    )

    N19_SMALL_ONLY_SWEEP_FRAMES.append(
        n19_run_small_only_case(
            n19_case_row,
            progress=True,
        )
    )

N19_SMALL_ONLY_SWEEP = pd.concat(
    N19_SMALL_ONLY_SWEEP_FRAMES,
    ignore_index=True,
).sort_values(
    [
        "size_ratio",
        "rank",
    ]
).reset_index(
    drop=True
)

display(
    N19_SMALL_ONLY_SWEEP[
        [
            "case_id",
            "size_ratio",
            "rank",
            "spacetime_relative_l2_error_percent",
            "time_averaged_l2_error_percent",
            "minimum_volume_fraction",
            "maximum_volume_fraction",
            "training_energy_percent",
        ]
    ]
)

N19_SMALL_ONLY_RANK_100 = (
    N19_SMALL_ONLY_SWEEP.loc[
        N19_SMALL_ONLY_SWEEP["rank"].eq(100)
    ]
    .sort_values("size_ratio")
    .reset_index(drop=True)
)

N19_ALL_PARTICLE_XZ_RANK_100_ROWS = []

for _, n19_case_row in (
    N19_CASE_PLAN.sort_values(
        "size_ratio"
    ).iterrows()
):
    n19_case_id = str(
        n19_case_row["case_id"]
    )

    n19_saved_path = (
        N19_REPOSITORY_ROOT
        / "results"
        / "tables"
        / "xz_particle_representation_sensitivity"
        / (
            "xz_particle_first_field_"
            f"{n19_case_id}.csv"
        )
    )

    assert n19_saved_path.is_file()

    n19_saved_table = pd.read_csv(
        n19_saved_path
    )

    n19_saved_rank_100 = (
        n19_saved_table.loc[
            n19_saved_table["rank"].eq(100)
        ]
        .iloc[0]
    )

    N19_ALL_PARTICLE_XZ_RANK_100_ROWS.append(
        {
            "case_id": n19_case_id,
            "size_ratio": float(
                n19_case_row["size_ratio"]
            ),
            "all_particle_xz_spacetime_error_percent":
                float(
                    n19_saved_rank_100[
                        "spacetime_relative_l2_error_percent"
                    ]
                ),
            "all_particle_xz_time_averaged_error_percent":
                float(
                    n19_saved_rank_100[
                        "time_averaged_l2_error_percent"
                    ]
                ),
        }
    )

N19_ALL_PARTICLE_XZ_RANK_100 = pd.DataFrame(
    N19_ALL_PARTICLE_XZ_RANK_100_ROWS
)

N19_FIVE_CASE_RANK_100_COMPARISON = (
    N19_SMALL_ONLY_RANK_100[
        [
            "case_id",
            "size_ratio",
            "spacetime_relative_l2_error_percent",
            "time_averaged_l2_error_percent",
        ]
    ]
    .rename(
        columns={
            "spacetime_relative_l2_error_percent":
                "small_only_spacetime_error_percent",
            "time_averaged_l2_error_percent":
                "small_only_time_averaged_error_percent",
        }
    )
    .merge(
        N19_ALL_PARTICLE_XZ_RANK_100,
        on=[
            "case_id",
            "size_ratio",
        ],
        how="inner",
        validate="one_to_one",
    )
    .sort_values("size_ratio")
    .reset_index(drop=True)
)

N19_FIVE_CASE_RANK_100_COMPARISON[
    "small_only_minus_all_particle_xz_percentage_points"
] = (
    N19_FIVE_CASE_RANK_100_COMPARISON[
        "small_only_spacetime_error_percent"
    ]
    - N19_FIVE_CASE_RANK_100_COMPARISON[
        "all_particle_xz_spacetime_error_percent"
    ]
)

display(
    N19_FIVE_CASE_RANK_100_COMPARISON
)

assert len(N19_SMALL_ONLY_SWEEP) == 25
assert len(
    N19_FIVE_CASE_RANK_100_COMPARISON
) == 5

assert np.isfinite(
    N19_SMALL_ONLY_SWEEP.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print(
    "Small-only five-case rows:",
    len(N19_SMALL_ONLY_SWEEP),
)

print(
    "Rank-100 comparison cases:",
    len(
        N19_FIVE_CASE_RANK_100_COMPARISON
    ),
)

print(
    "Files written:",
    False,
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "Notebook 19 five-case small-only sweep passed:",
    True,
)


Running small-only case: ratio_1_252525_num_26
ratio_1_252525_num_26 true validation: 25 of 99
ratio_1_252525_num_26 true validation: 50 of 99
ratio_1_252525_num_26 true validation: 75 of 99
ratio_1_252525_num_26 true validation: 99 of 99
ratio_1_252525_num_26 rank 0: 25 of 99
ratio_1_252525_num_26 rank 0: 50 of 99
ratio_1_252525_num_26 rank 0: 75 of 99
ratio_1_252525_num_26 rank 0: 99 of 99
ratio_1_252525_num_26 rank 8: 25 of 99
ratio_1_252525_num_26 rank 8: 50 of 99
ratio_1_252525_num_26 rank 8: 75 of 99
ratio_1_252525_num_26 rank 8: 99 of 99
ratio_1_252525_num_26 rank 14: 25 of 99
ratio_1_252525_num_26 rank 14: 50 of 99
ratio_1_252525_num_26 rank 14: 75 of 99
ratio_1_252525_num_26 rank 14: 99 of 99
ratio_1_252525_num_26 rank 27: 25 of 99
ratio_1_252525_num_26 rank 27: 50 of 99
ratio_1_252525_num_26 rank 27: 75 of 99
ratio_1_252525_num_26 rank 27: 99 of 99
ratio_1_252525_num_26 rank 100: 25 of 99
ratio_1_252525_num_26 rank 100: 50 of 99
ratio_1_252525_num_26 rank 100: 75 of 99
ratio

,case_id,size_ratio,rank,spacetime_relative_l2_error_percent,time_averaged_l2_error_percent,minimum_volume_fraction,maximum_volume_fraction,training_energy_percent
0,ratio_1_000000_num_1,1.000000,0,577.294936,580.725546,0.0,17.231289,0.000000
1,ratio_1_000000_num_1,1.000000,8,103.364067,84.921304,0.0,2.025867,96.000163
2,ratio_1_000000_num_1,1.000000,14,101.160475,80.639671,0.0,2.372336,99.121420
3,ratio_1_000000_num_1,1.000000,27,98.911327,78.512980,0.0,2.266410,99.942334
4,ratio_1_000000_num_1,1.000000,100,93.301591,74.816481,0.0,2.042362,100.000000
5,ratio_1_252525_num_26,1.252525,0,515.590935,517.822651,0.0,14.928934,0.000000
6,ratio_1_252525_num_26,1.252525,8,139.168499,111.697620,0.0,4.703957,95.580357
7,ratio_1_252525_num_26,1.252525,14,135.733817,106.147982,0.0,5.543218,99.074868
8,ratio_1_252525_num_26,1.252525,27,133.811712,103.581332,0.0,5.816625,99.937239
9,ratio_1_252525_num_26,1.252525,100,126.138384,99.350178,0.0,5.186418,100.000000


,case_id,size_ratio,small_only_spacetime_error_percent,small_only_time_averaged_error_percent,all_particle_xz_spacetime_error_percent,all_particle_xz_time_averaged_error_percent,small_only_minus_all_particle_xz_percentage_points
0,ratio_1_000000_num_1,1.000000,93.301591,74.816481,92.687320,73.646940,0.614271
1,ratio_1_252525_num_26,1.252525,126.138384,99.350178,102.487379,79.538501,23.651005
2,ratio_1_494949_num_50,1.494949,145.306639,111.827583,123.800094,95.458701,21.506545
3,ratio_1_747475_num_75,1.747475,145.034964,112.593435,133.904401,104.181047,11.130563
4,ratio_2_000000_num_100,2.000000,145.613673,111.848316,140.358939,108.150767,5.254734


Small-only five-case rows: 25
Rank-100 comparison cases: 5
Files written: False
Raw DEM files accessed: False
Notebook 19 five-case small-only sweep passed: True


## Five-case observable-matched POD sweep

The pilot result is extended to the remaining four frozen size-ratio cases.

For every case, the same protocol is retained:

- 101 training snapshots;
- training-mean centring;
- validation projection onto the training POD basis;
- ranks $0,8,14,27,100$;
- direct small-species coarse-graining of reconstructed $XZ$ coordinates.

The purpose is not to introduce a new model-selection procedure. It is to determine whether the pilot conclusion persists across the frozen parameter range.

In [9]:
def n19_run_small_only_case(
    case_row,
    ranks=(0, 8, 14, 27, 100),
    progress=True,
):
    case_data = n19_load_compact_case(
        case_row
    )

    case_id = str(
        case_row["case_id"]
    )

    size_ratio = float(
        case_row["size_ratio"]
    )

    positions_small_xz = (
        case_data["positions_small_xz"]
    )

    training_indices = (
        case_data["training_indices"]
    )

    validation_indices = (
        case_data["validation_indices"]
    )

    small_particle_volumes = (
        case_data["small_particle_volumes"]
    )

    number_of_small_particles = (
        positions_small_xz.shape[1]
    )

    training_matrix = (
        positions_small_xz[
            training_indices
        ]
        .reshape(
            len(training_indices),
            -1,
        )
        .T
    )

    validation_matrix = (
        positions_small_xz[
            validation_indices
        ]
        .reshape(
            len(validation_indices),
            -1,
        )
        .T
    )

    training_mean = (
        training_matrix.mean(
            axis=1,
            keepdims=True,
        )
    )

    training_centered = (
        training_matrix
        - training_mean
    )

    validation_centered = (
        validation_matrix
        - training_mean
    )

    (
        pod_modes,
        singular_values,
        right_singular_vectors_transpose,
        cumulative_energy,
        numerical_rank,
    ) = n19_calculate_thin_pod(
        training_centered
    )

    assert numerical_rank >= max(
        ranks
    )

    true_validation_fields = (
        n19_coarse_grain_small_xz_sequence(
            positions_small_xz[
                validation_indices
            ],
            small_particle_volumes,
            label=(
                f"{case_id} true validation:"
                if progress
                else None
            ),
        )
    )

    result_rows = []

    for rank in ranks:
        if rank == 0:
            reconstructed_matrix = (
                np.repeat(
                    training_mean,
                    len(validation_indices),
                    axis=1,
                )
            )
        else:
            basis = pod_modes[
                :,
                :rank,
            ]

            validation_coefficients = (
                basis.T
                @ validation_centered
            )

            reconstructed_matrix = (
                training_mean
                + basis
                @ validation_coefficients
            )

        reconstructed_small_xz = (
            reconstructed_matrix.T.reshape(
                len(validation_indices),
                number_of_small_particles,
                2,
            )
        )

        reconstructed_fields = (
            n19_coarse_grain_small_xz_sequence(
                reconstructed_small_xz,
                small_particle_volumes,
                label=(
                    f"{case_id} rank {rank}:"
                    if progress
                    else None
                ),
            )
        )

        metrics = (
            n19_field_summary_metrics(
                reconstructed_fields,
                true_validation_fields,
                rank,
            )
        )

        metrics.update(
            {
                "case_id": case_id,
                "size_ratio": size_ratio,
                "pod_numerical_rank": int(
                    numerical_rank
                ),
                "training_snapshot_count": int(
                    len(training_indices)
                ),
                "validation_snapshot_count": int(
                    len(validation_indices)
                ),
                "training_energy_percent": (
                    0.0
                    if rank == 0
                    else 100.0
                    * cumulative_energy[
                        rank - 1
                    ]
                ),
            }
        )

        result_rows.append(
            metrics
        )

    return pd.DataFrame(
        result_rows
    ).sort_values(
        "rank"
    ).reset_index(
        drop=True
    )


N19_SMALL_ONLY_SWEEP_FRAMES = [
    N19_SMALL_ONLY_PILOT_RESULTS.assign(
        case_id=N19_PILOT_CASE_ID,
        size_ratio=1.0,
        pod_numerical_rank=(
            N19_SMALL_POD_NUMERICAL_RANK
        ),
        training_snapshot_count=101,
        validation_snapshot_count=99,
    )
]

for _, n19_case_row in (
    N19_CASE_PLAN.sort_values(
        "size_ratio"
    ).iterrows()
):
    if (
        str(
            n19_case_row["case_id"]
        )
        == N19_PILOT_CASE_ID
    ):
        continue

    print(
        "\nRunning small-only case:",
        n19_case_row["case_id"],
    )

    n19_case_results = (
        n19_run_small_only_case(
            n19_case_row,
            progress=True,
        )
    )

    N19_SMALL_ONLY_SWEEP_FRAMES.append(
        n19_case_results
    )

N19_SMALL_ONLY_SWEEP = pd.concat(
    N19_SMALL_ONLY_SWEEP_FRAMES,
    ignore_index=True,
)

N19_SMALL_ONLY_SWEEP = (
    N19_SMALL_ONLY_SWEEP.sort_values(
        [
            "size_ratio",
            "rank",
        ]
    )
    .reset_index(drop=True)
)

display(
    N19_SMALL_ONLY_SWEEP[
        [
            "case_id",
            "size_ratio",
            "rank",
            "spacetime_relative_l2_error_percent",
            "time_averaged_l2_error_percent",
            "minimum_volume_fraction",
            "maximum_volume_fraction",
            "training_energy_percent",
        ]
    ].style.format(
        {
            "size_ratio": "{:.6f}",
            "spacetime_relative_l2_error_percent":
                "{:.6f}",
            "time_averaged_l2_error_percent":
                "{:.6f}",
            "minimum_volume_fraction":
                "{:.6e}",
            "maximum_volume_fraction":
                "{:.6e}",
            "training_energy_percent":
                "{:.6f}",
        }
    )
)

N19_SMALL_ONLY_RANK_100 = (
    N19_SMALL_ONLY_SWEEP.loc[
        N19_SMALL_ONLY_SWEEP[
            "rank"
        ].eq(100)
    ]
    .sort_values("size_ratio")
    .reset_index(drop=True)
)

N19_ALL_PARTICLE_XZ_RANK_100_ROWS = []

for _, n19_case_row in (
    N19_CASE_PLAN.sort_values(
        "size_ratio"
    ).iterrows()
):
    n19_case_id = str(
        n19_case_row["case_id"]
    )

    n19_saved_path = (
        N19_REPOSITORY_ROOT
        / "results"
        / "tables"
        / "xz_particle_representation_sensitivity"
        / (
            "xz_particle_first_field_"
            f"{n19_case_id}.csv"
        )
    )

    assert n19_saved_path.is_file()

    n19_saved_table = pd.read_csv(
        n19_saved_path
    )

    n19_saved_rank_100 = (
        n19_saved_table.loc[
            n19_saved_table[
                "rank"
            ].eq(100)
        ]
        .iloc[0]
    )

    N19_ALL_PARTICLE_XZ_RANK_100_ROWS.append(
        {
            "case_id": n19_case_id,
            "size_ratio": float(
                n19_case_row["size_ratio"]
            ),
            "all_particle_xz_spacetime_error_percent":
                float(
                    n19_saved_rank_100[
                        "spacetime_relative_l2_error_percent"
                    ]
                ),
            "all_particle_xz_time_averaged_error_percent":
                float(
                    n19_saved_rank_100[
                        "time_averaged_l2_error_percent"
                    ]
                ),
        }
    )

N19_ALL_PARTICLE_XZ_RANK_100 = pd.DataFrame(
    N19_ALL_PARTICLE_XZ_RANK_100_ROWS
)

N19_FIVE_CASE_RANK_100_COMPARISON = (
    N19_SMALL_ONLY_RANK_100[
        [
            "case_id",
            "size_ratio",
            "spacetime_relative_l2_error_percent",
            "time_averaged_l2_error_percent",
        ]
    ]
    .rename(
        columns={
            "spacetime_relative_l2_error_percent":
                "small_only_spacetime_error_percent",
            "time_averaged_l2_error_percent":
                "small_only_time_averaged_error_percent",
        }
    )
    .merge(
        N19_ALL_PARTICLE_XZ_RANK_100,
        on=[
            "case_id",
            "size_ratio",
        ],
        how="inner",
        validate="one_to_one",
    )
    .sort_values("size_ratio")
    .reset_index(drop=True)
)

N19_FIVE_CASE_RANK_100_COMPARISON[
    "small_only_minus_all_particle_xz_percentage_points"
] = (
    N19_FIVE_CASE_RANK_100_COMPARISON[
        "small_only_spacetime_error_percent"
    ]
    - N19_FIVE_CASE_RANK_100_COMPARISON[
        "all_particle_xz_spacetime_error_percent"
    ]
)

display(
    N19_FIVE_CASE_RANK_100_COMPARISON.style.format(
        {
            "size_ratio": "{:.6f}",
            "small_only_spacetime_error_percent":
                "{:.6f}",
            "small_only_time_averaged_error_percent":
                "{:.6f}",
            "all_particle_xz_spacetime_error_percent":
                "{:.6f}",
            "all_particle_xz_time_averaged_error_percent":
                "{:.6f}",
            "small_only_minus_all_particle_xz_percentage_points":
                "{:.6f}",
        }
    )
)

assert len(
    N19_SMALL_ONLY_SWEEP
) == 5 * 5

assert (
    set(
        N19_SMALL_ONLY_SWEEP[
            "case_id"
        ]
    )
    == set(
        N19_CASE_PLAN[
            "case_id"
        ]
    )
)

assert (
    set(
        N19_SMALL_ONLY_SWEEP[
            "rank"
        ]
    )
    == {
        0,
        8,
        14,
        27,
        100,
    }
)

assert (
    N19_SMALL_ONLY_SWEEP[
        "training_snapshot_count"
    ]
    == 101
).all()

assert (
    N19_SMALL_ONLY_SWEEP[
        "validation_snapshot_count"
    ]
    == 99
).all()

assert np.isfinite(
    N19_SMALL_ONLY_SWEEP.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print(
    "Small-only five-case rows:",
    len(N19_SMALL_ONLY_SWEEP),
)

print(
    "Rank-100 comparison cases:",
    len(
        N19_FIVE_CASE_RANK_100_COMPARISON
    ),
)

print(
    "Rank-100 small-only errors:",
    N19_FIVE_CASE_RANK_100_COMPARISON[
        "small_only_spacetime_error_percent"
    ].round(6).tolist(),
)

print(
    "Rank-100 all-particle XZ errors:",
    N19_FIVE_CASE_RANK_100_COMPARISON[
        "all_particle_xz_spacetime_error_percent"
    ].round(6).tolist(),
)

print(
    "Files written:",
    False,
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "Notebook 19 five-case small-only sweep passed:",
    True,
)


Running small-only case: ratio_1_252525_num_26
ratio_1_252525_num_26 true validation: 25 of 99
ratio_1_252525_num_26 true validation: 50 of 99
ratio_1_252525_num_26 true validation: 75 of 99
ratio_1_252525_num_26 true validation: 99 of 99
ratio_1_252525_num_26 rank 0: 25 of 99
ratio_1_252525_num_26 rank 0: 50 of 99
ratio_1_252525_num_26 rank 0: 75 of 99
ratio_1_252525_num_26 rank 0: 99 of 99
ratio_1_252525_num_26 rank 8: 25 of 99
ratio_1_252525_num_26 rank 8: 50 of 99
ratio_1_252525_num_26 rank 8: 75 of 99
ratio_1_252525_num_26 rank 8: 99 of 99
ratio_1_252525_num_26 rank 14: 25 of 99
ratio_1_252525_num_26 rank 14: 50 of 99
ratio_1_252525_num_26 rank 14: 75 of 99
ratio_1_252525_num_26 rank 14: 99 of 99
ratio_1_252525_num_26 rank 27: 25 of 99
ratio_1_252525_num_26 rank 27: 50 of 99
ratio_1_252525_num_26 rank 27: 75 of 99
ratio_1_252525_num_26 rank 27: 99 of 99
ratio_1_252525_num_26 rank 100: 25 of 99
ratio_1_252525_num_26 rank 100: 50 of 99
ratio_1_252525_num_26 rank 100: 75 of 99
ratio

,case_id,size_ratio,rank,spacetime_relative_l2_error_percent,time_averaged_l2_error_percent,minimum_volume_fraction,maximum_volume_fraction,training_energy_percent
0,ratio_1_000000_num_1,1.000000,0,577.294936,580.725546,0.000000e+00,1.723129e+01,0.000000
1,ratio_1_000000_num_1,1.000000,8,103.364067,84.921304,0.000000e+00,2.025867e+00,96.000163
2,ratio_1_000000_num_1,1.000000,14,101.160475,80.639671,0.000000e+00,2.372336e+00,99.121420
3,ratio_1_000000_num_1,1.000000,27,98.911327,78.512980,0.000000e+00,2.266410e+00,99.942334
4,ratio_1_000000_num_1,1.000000,100,93.301591,74.816481,0.000000e+00,2.042362e+00,100.000000
5,ratio_1_252525_num_26,1.252525,0,515.590935,517.822651,0.000000e+00,1.492893e+01,0.000000
6,ratio_1_252525_num_26,1.252525,8,139.168499,111.697620,0.000000e+00,4.703957e+00,95.580357
7,ratio_1_252525_num_26,1.252525,14,135.733817,106.147982,0.000000e+00,5.543218e+00,99.074868
8,ratio_1_252525_num_26,1.252525,27,133.811712,103.581332,0.000000e+00,5.816625e+00,99.937239
9,ratio_1_252525_num_26,1.252525,100,126.138384,99.350178,0.000000e+00,5.186418e+00,100.000000


,case_id,size_ratio,small_only_spacetime_error_percent,small_only_time_averaged_error_percent,all_particle_xz_spacetime_error_percent,all_particle_xz_time_averaged_error_percent,small_only_minus_all_particle_xz_percentage_points
0,ratio_1_000000_num_1,1.000000,93.301591,74.816481,92.687320,73.646940,0.614271
1,ratio_1_252525_num_26,1.252525,126.138384,99.350178,102.487379,79.538501,23.651005
2,ratio_1_494949_num_50,1.494949,145.306639,111.827583,123.800094,95.458701,21.506545
3,ratio_1_747475_num_75,1.747475,145.034964,112.593435,133.904401,104.181047,11.130563
4,ratio_2_000000_num_100,2.000000,145.613673,111.848316,140.358939,108.150767,5.254734


Small-only five-case rows: 25
Rank-100 comparison cases: 5
Rank-100 small-only errors: [93.301591, 126.138384, 145.306639, 145.034964, 145.613673]
Rank-100 all-particle XZ errors: [92.68732, 102.487379, 123.800094, 133.904401, 140.358939]
Files written: False
Raw DEM files accessed: False
Notebook 19 five-case small-only sweep passed: True


# Notebook 19 — findings and limitations

The observable-matched representation restricted the particle POD to the coordinates that directly influence the measured small-species volume-fraction field: the small-particle $XZ$ coordinates.

For the equal-size pilot, the small-only representation produced a rank-$100$ validation error of

$$
93.30\%,
$$

compared with

$$
92.69\%
$$

for the previously saved all-particle-$XZ$ baseline. The rank-zero results agreed exactly, confirming that the two implementations use the same mean small-particle state and coarse-graining operator.

The five-case sweep showed that the small-only representation did not improve the particle-first route across the parameter range. At rank $100$, the small-only space-time errors were

$$
93.30\%,\quad
126.14\%,\quad
145.31\%,\quad
145.03\%,\quad
145.61\%
$$

for size ratios

$$
1.000000,\quad
1.252525,\quad
1.494949,\quad
1.747475,\quad
2.000000.
$$

The corresponding all-particle-$XZ$ errors were

$$
92.69\%,\quad
102.49\%,\quad
123.80\%,\quad
133.90\%,\quad
140.36\%.
$$

Thus, removing coordinates associated with the large species does not account for the poor particle-first performance. The observable-matched representation is slightly worse in every tested case.

The row-continuity audit provides an important qualification. The compact caches do not contain persistent particle identifiers, and the stored row associated with a particle was the nearest same-species successor in only approximately $1.4$–$5.9\%$ of adjacent snapshot comparisons. The median stored-row displacement was also several times larger than the median nearest same-species distance. Consequently, the particle-coordinate POD should be interpreted as a POD of labelled array rows, not as a physically tracked particle-trajectory model.

This result does not prove that every possible particle-based reduction is ineffective. A physically different study would require persistent particle tracking, permutation-invariant features, local-neighbourhood descriptors, or a suitable transport-aware representation. Those alternatives are outside the bounded scope of this dissertation.

The additional experiment therefore strengthens the main conclusion in a negative but useful way: for the supplied compact caches and the tested labelled-coordinate formulations, coarse-graining before POD is substantially more accurate and physically interpretable than POD before coarse-graining. The conclusion is specifically about the tested small-species volume-fraction observable and should not automatically be generalized to velocity, momentum, stress or other vector-valued fields.

## Notebook 19 artifact export

The validated Notebook 19 results are now exported as reproducible CSV tables and vector/raster figures.

The manifest records the trusted Notebook 12 and Notebook 13 manifests together with the newly generated Notebook 19 artifacts. The manifest excludes itself from its own hash records.

In [10]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt


N19_OUTPUT_TABLE_DIRECTORY = (
    N19_REPOSITORY_ROOT
    / "results"
    / "tables"
    / "observable_matched_particle_pod"
)

N19_OUTPUT_FIGURE_DIRECTORY = (
    N19_REPOSITORY_ROOT
    / "results"
    / "figures"
    / "observable_matched_particle_pod"
)

N19_OUTPUT_TABLE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

N19_OUTPUT_FIGURE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


def n19_hash_file(
    path,
    chunk_size=8 * 1024**2,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


# Compact audit tables.
N19_PILOT_RADIUS_AUDIT = pd.DataFrame(
    [
        {
            "case_id": N19_PILOT_CASE_ID,
            "small_radius_minimum": float(
                N19_PILOT_CASE[
                    "small_radii"
                ].min()
            ),
            "small_radius_median": float(
                N19_PILOT_CASE[
                    "small_radius"
                ]
            ),
            "small_radius_maximum": float(
                N19_PILOT_CASE[
                    "small_radii"
                ].max()
            ),
            "large_radius_minimum": float(
                N19_PILOT_CASE[
                    "particle_radii"
                ][
                    N19_PILOT_CASE[
                        "large_mask"
                    ]
                ].min()
            ),
            "large_radius_median": float(
                N19_PILOT_CASE[
                    "large_radius"
                ]
            ),
            "large_radius_maximum": float(
                N19_PILOT_CASE[
                    "particle_radii"
                ][
                    N19_PILOT_CASE[
                        "large_mask"
                    ]
                ].max()
            ),
            "representative_small_diameter": float(
                N19_PILOT_CASE[
                    "small_diameter"
                ]
            ),
            "lucy_support_to_small_diameter": float(
                N19_PILOT_CASE[
                    "lucy_support_to_small_diameter"
                ]
            ),
        }
    ]
)


N19_TABLES_TO_SAVE = {
    "small_only_pod_sweep": (
        N19_SMALL_ONLY_SWEEP
    ),
    "rank100_comparison": (
        N19_FIVE_CASE_RANK_100_COMPARISON
    ),
    "row_continuity_summary": (
        N19_CONTINUITY_SUMMARY
    ),
    "operator_regression": (
        N19_OPERATOR_REGRESSION
    ),
    "pilot_radius_audit": (
        N19_PILOT_RADIUS_AUDIT
    ),
}

N19_SAVED_TABLE_PATHS = []

for table_name, table in (
    N19_TABLES_TO_SAVE.items()
):
    table_path = (
        N19_OUTPUT_TABLE_DIRECTORY
        / f"{table_name}.csv"
    )

    table.to_csv(
        table_path,
        index=False,
    )

    N19_SAVED_TABLE_PATHS.append(
        table_path
    )


# Figure 1: five-case rank-100 comparison.
fig, ax = plt.subplots(
    figsize=(7.2, 4.8)
)

ax.plot(
    N19_FIVE_CASE_RANK_100_COMPARISON[
        "size_ratio"
    ],
    N19_FIVE_CASE_RANK_100_COMPARISON[
        "small_only_spacetime_error_percent"
    ],
    marker="o",
    linewidth=2.0,
    label="Small-only $XZ$ POD",
)

ax.plot(
    N19_FIVE_CASE_RANK_100_COMPARISON[
        "size_ratio"
    ],
    N19_FIVE_CASE_RANK_100_COMPARISON[
        "all_particle_xz_spacetime_error_percent"
    ],
    marker="s",
    linewidth=2.0,
    label="All-particle $XZ$ POD",
)

ax.set_xlabel("Particle-size ratio")
ax.set_ylabel(
    "Rank-100 validation error (%)"
)
ax.set_title(
    "Observable-matched versus all-particle $XZ$ POD"
)
ax.grid(
    alpha=0.3
)
ax.legend()
fig.tight_layout()

N19_RANK100_FIGURE_PATHS = [
    N19_OUTPUT_FIGURE_DIRECTORY
    / "small_only_vs_all_particle_xz_rank100.pdf",
    N19_OUTPUT_FIGURE_DIRECTORY
    / "small_only_vs_all_particle_xz_rank100.png",
]

for figure_path in N19_RANK100_FIGURE_PATHS:
    fig.savefig(
        figure_path,
        bbox_inches="tight",
    )

plt.close(fig)


# Figure 2: small-only error by rank and size ratio.
fig, ax = plt.subplots(
    figsize=(7.2, 4.8)
)

for size_ratio, group in (
    N19_SMALL_ONLY_SWEEP.groupby(
        "size_ratio"
    )
):
    group = group.sort_values("rank")

    ax.plot(
        group["rank"],
        group[
            "spacetime_relative_l2_error_percent"
        ],
        marker="o",
        linewidth=1.8,
        label=f"$\\lambda={size_ratio:.3f}$",
    )

ax.set_xlabel("POD rank")
ax.set_ylabel(
    "Validation space-time error (%)"
)
ax.set_title(
    "Small-species-only $XZ$ POD sweep"
)
ax.grid(
    alpha=0.3
)
ax.legend(
    title="Size ratio",
    fontsize=8,
)
fig.tight_layout()

N19_SWEEP_FIGURE_PATHS = [
    N19_OUTPUT_FIGURE_DIRECTORY
    / "small_only_pod_rank_sweep.pdf",
    N19_OUTPUT_FIGURE_DIRECTORY
    / "small_only_pod_rank_sweep.png",
]

for figure_path in N19_SWEEP_FIGURE_PATHS:
    fig.savefig(
        figure_path,
        bbox_inches="tight",
    )

plt.close(fig)


# Figure 3: row-continuity diagnostic.
fig, ax = plt.subplots(
    figsize=(7.2, 4.8)
)

continuity_plot = (
    N19_CONTINUITY_SUMMARY.copy()
)

continuity_plot["species_name"] = (
    continuity_plot["species"]
    .map(
        {
            0: "Large species",
            1: "Small species",
        }
    )
)

for species_name, group in (
    continuity_plot.groupby(
        "species_name"
    )
):
    group = group.sort_values(
        "size_ratio"
    )

    ax.plot(
        group["size_ratio"],
        100.0
        * group[
            "mean_stored_partner_nearest_fraction"
        ],
        marker="o",
        linewidth=1.8,
        label=species_name,
    )

ax.set_xlabel("Particle-size ratio")
ax.set_ylabel(
    "Stored row nearest-neighbour fraction (%)"
)
ax.set_title(
    "Particle-row continuity diagnostic"
)
ax.set_ylim(
    bottom=0.0
)
ax.grid(
    alpha=0.3
)
ax.legend()
fig.tight_layout()

N19_CONTINUITY_FIGURE_PATHS = [
    N19_OUTPUT_FIGURE_DIRECTORY
    / "row_continuity_diagnostic.pdf",
    N19_OUTPUT_FIGURE_DIRECTORY
    / "row_continuity_diagnostic.png",
]

for figure_path in N19_CONTINUITY_FIGURE_PATHS:
    fig.savefig(
        figure_path,
        bbox_inches="tight",
    )

plt.close(fig)


N19_SAVED_FIGURE_PATHS = (
    N19_RANK100_FIGURE_PATHS
    + N19_SWEEP_FIGURE_PATHS
    + N19_CONTINUITY_FIGURE_PATHS
)


# Trusted input manifests.
N19_NOTEBOOK12_MANIFEST_PATH = (
    N19_REPOSITORY_ROOT
    / "results"
    / "tables"
    / "multi_parameter_route_order"
    / "multi_parameter_route_order_manifest.json"
)

N19_NOTEBOOK13_MANIFEST_PATH = (
    N19_REPOSITORY_ROOT
    / "results"
    / "tables"
    / "xz_particle_representation_sensitivity"
    / "xz_particle_representation_sensitivity_manifest.json"
)

assert N19_NOTEBOOK12_MANIFEST_PATH.is_file()
assert N19_NOTEBOOK13_MANIFEST_PATH.is_file()


N19_MANIFEST_PATH = (
    N19_OUTPUT_TABLE_DIRECTORY
    / "notebook19_observable_matched_manifest.json"
)


N19_RECORD_PATHS = [
    N19_NOTEBOOK12_MANIFEST_PATH,
    N19_NOTEBOOK13_MANIFEST_PATH,
    *N19_SAVED_TABLE_PATHS,
    *N19_SAVED_FIGURE_PATHS,
]


N19_RECORDS = []

for record_path in N19_RECORD_PATHS:
    record_path = Path(record_path)

    assert record_path.is_file()

    if record_path.suffix.lower() == ".csv":
        role = "table_artifact"
    elif record_path.suffix.lower() in {
        ".pdf",
        ".png",
    }:
        role = "figure_artifact"
    else:
        role = "trusted_input"

    N19_RECORDS.append(
        {
            "relative_path": str(
                record_path.relative_to(
                    N19_REPOSITORY_ROOT
                )
            ),
            "role": role,
            "size_bytes": int(
                record_path.stat().st_size
            ),
            "sha256": n19_hash_file(
                record_path
            ),
        }
    )


N19_MANIFEST = {
    "schema_version": 1,
    "manifest_type": (
        "Notebook 19 observable-matched "
        "particle POD and row-continuity audit"
    ),
    "hash_algorithm": "SHA-256",
    "recorded_file_count": len(
        N19_RECORDS
    ),
    "records": N19_RECORDS,
    "scope": {
        "raw_dem_files_accessed": False,
        "external_cases_included": False,
        "numerical_particle_arrays_loaded": True,
        "files_written": True,
        "manifest_self_hash_excluded": True,
    },
    "frozen_protocol": {
        "case_count": 5,
        "ranks": [
            0,
            8,
            14,
            27,
            100,
        ],
        "training_snapshot_count": 101,
        "validation_snapshot_count": 99,
        "grid_shape": [
            100,
            100,
        ],
        "lucy_cutoff": 3.0,
        "axial_averaging_length": 10.0,
        "small_species": 1,
        "row_continuity_interval_count": 200,
    },
    "key_findings": {
        "pilot_rank100_small_only_error_percent": float(
            N19_SMALL_ONLY_COMPARISON.loc[
                N19_SMALL_ONLY_COMPARISON[
                    "rank"
                ].eq(100)
            ][
                "small_only_spacetime_error_percent"
            ].iloc[0]
        ),
        "pilot_rank100_all_particle_xz_error_percent":
            float(
                N19_SMALL_ONLY_COMPARISON.loc[
                    N19_SMALL_ONLY_COMPARISON[
                        "rank"
                    ].eq(100)
                ][
                    "all_particle_xz_spacetime_error_percent"
                ].iloc[0]
            ),
        "minimum_row_nearest_fraction": float(
            N19_CONTINUITY_SUMMARY[
                "minimum_stored_partner_nearest_fraction"
            ].min()
        ),
        "mean_row_nearest_fraction": float(
            N19_CONTINUITY_SUMMARY[
                "mean_stored_partner_nearest_fraction"
            ].mean()
        ),
    },
}

with N19_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        N19_MANIFEST,
        handle,
        indent=2,
    )


# Re-read and independently verify the generated records.
with N19_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    N19_RELOADED_MANIFEST = json.load(
        handle
    )

assert (
    N19_RELOADED_MANIFEST[
        "recorded_file_count"
    ]
    == len(N19_RECORDS)
)

for record in N19_RELOADED_MANIFEST[
    "records"
]:
    record_path = (
        N19_REPOSITORY_ROOT
        / record["relative_path"]
    )

    assert record_path.is_file()
    assert (
        record_path.stat().st_size
        == record["size_bytes"]
    )
    assert (
        n19_hash_file(record_path)
        == record["sha256"]
    )

assert N19_MANIFEST_PATH.is_file()
assert N19_MANIFEST_PATH not in [
    N19_REPOSITORY_ROOT
    / record["relative_path"]
    for record in N19_RELOADED_MANIFEST[
        "records"
    ]
]

print(
    "Saved Notebook 19 tables:",
    len(N19_SAVED_TABLE_PATHS),
)

for table_path in N19_SAVED_TABLE_PATHS:
    print(table_path)

print(
    "Saved Notebook 19 figures:",
    len(N19_SAVED_FIGURE_PATHS),
)

for figure_path in N19_SAVED_FIGURE_PATHS:
    print(figure_path)

print(
    "Notebook 19 manifest:",
    N19_MANIFEST_PATH,
)

print(
    "Recorded artifacts:",
    len(N19_RECORDS),
)

print(
    "Mean stored-partner-nearest fraction:",
    float(
        N19_MANIFEST[
            "key_findings"
        ][
            "mean_row_nearest_fraction"
        ]
    ),
)

print(
    "Minimum stored-partner-nearest fraction:",
    float(
        N19_MANIFEST[
            "key_findings"
        ][
            "minimum_row_nearest_fraction"
        ]
    ),
)

print(
    "Raw DEM files accessed:",
    False,
)

print(
    "External cases included:",
    False,
)

print(
    "Notebook 19 artifacts saved and validated:",
    True,
)

Saved Notebook 19 tables: 5
/Users/rallen/Documents/msc-rom-particle-systems/results/tables/observable_matched_particle_pod/small_only_pod_sweep.csv
/Users/rallen/Documents/msc-rom-particle-systems/results/tables/observable_matched_particle_pod/rank100_comparison.csv
/Users/rallen/Documents/msc-rom-particle-systems/results/tables/observable_matched_particle_pod/row_continuity_summary.csv
/Users/rallen/Documents/msc-rom-particle-systems/results/tables/observable_matched_particle_pod/operator_regression.csv
/Users/rallen/Documents/msc-rom-particle-systems/results/tables/observable_matched_particle_pod/pilot_radius_audit.csv
Saved Notebook 19 figures: 6
/Users/rallen/Documents/msc-rom-particle-systems/results/figures/observable_matched_particle_pod/small_only_vs_all_particle_xz_rank100.pdf
/Users/rallen/Documents/msc-rom-particle-systems/results/figures/observable_matched_particle_pod/small_only_vs_all_particle_xz_rank100.png
/Users/rallen/Documents/msc-rom-particle-systems/results/figure